# Notebook 07 — SCS artwork and tables

**Purpose.** Every figure and table of the *Sustainable Cities and Society* (SCS) article and of its
supplementary material, built from one place and in manuscript order.

**Manuscript.** SCS v4, in preparation. Numbering from the work plan v3.2.

**Data.** The published dataset v1.0.1 (DOI [10.5281/zenodo.23777933](https://doi.org/10.5281/zenodo.23777933); all versions:
10.5281/zenodo.23050559), read from
`Results/05_hotspot_analysis/surface_thermal_anomaly_stakeholder_composite/`. Association results
(NB06) are the one declared exception.

**Format reference.** The v3.0 outputs in `Results/manuscript_outputs/_superseded_v3.0/`, opened only
as pictures for the side-by-side previews, never read as data.

**How to run.** `python scripts/build_scs_outputs.py`, or Run All. No GEE, EUMETSAT or OSM call is made.

**Rules.**
1. Values from the package; every shown number asserted with `check()`.
2. Caches only for what the package does not hold, and only for the items the plan lists.
3. Every read goes through `read_source()`; a forbidden source stops the run.
4. v3.0 formats, no merges or new layouts; print sizes are reported, never forced.
5. No titles inside figures; panel letters stay; cluster codes C1 … C25.

**Monitoring.** [Work plan](../docs/scs_rebuild/SCS_Figures_Tables_WorkPlan.md) ·
[INDEX](../Results/manuscript_outputs/_preview/INDEX.md) (previews, sources, checks) ·
[WORKLOG](../docs/scs_rebuild/WORKLOG.md)

## 0 · Setup

Paths, the package reader `pkg()`, assertions `check()`, the checked `savefig()` (explicit `format=`,
magic bytes), approved cluster labels, the data-source guard `read_source()` with its provenance log
`_data_sources.csv`, the item marker `begin()`, and the monitoring helpers `compare_preview()` and
`fig_report()`. The second cell rebuilds the shared inputs (zone geometries, WUP table, typology map).

In [ ]:
import sys, shutil, warnings, json, re, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import xlsxwriter
from shapely.geometry import Point
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

# repo root = parent of this /Notebook folder
ROOT = Path.cwd()
if ROOT.name.lower() == "notebook":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
import config
import visualization as viz
import scs_style as S          # shared: approved cluster labels, colours, checked savefig
# The plotting modules in src/ are reloaded before every cell, so a cell run on its own in a
# long-running kernel always draws with the current code.
_ip = get_ipython()
if "autoreload" not in _ip.extension_manager.loaded:
    _ip.run_line_magic("load_ext", "autoreload")
_ip.run_line_magic("autoreload", "1")
_ip.run_line_magic("aimport", "visualization, scs_style, temporal_panels, pub_hotspot_tables")

warnings.filterwarnings("ignore", message=".*FancyArrowPatch.*")
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight",
                     "pdf.fonttype": 42, "ps.fonttype": 42})

# ── input caches ──
RESULTS_AOI = ROOT / "Results" / "01_aoi_delineation"
RESULTS_03  = ROOT / "Results" / "03_temporal_analysis"
RESULTS_04  = ROOT / "Results" / "04_temporal_analysis_daily"
RESULTS_05  = ROOT / "Results" / "05_hotspot_analysis"
RESULTS_06  = ROOT / "Results" / "06_hotspot_modelling"
STUDY_YEAR  = config.PROJECT_STUDY_YEAR

# ── the published v1.0.1 package: the source of record for every value ──
PKG  = RESULTS_05 / "surface_thermal_anomaly_stakeholder_composite"
GPKG = PKG / "surface_thermal_anomaly_stakeholder_composite.gpkg"
assert GPKG.exists(), f"build the package first: {GPKG}"

_LAYERS = {}


def pkg(layer, geom=False):
    """Read a layer of the published package, once (every call is logged)."""
    read_source(GPKG, layer=layer)
    key = (layer, geom)
    if key not in _LAYERS:
        _LAYERS[key] = (gpd.read_file(GPKG, layer=layer) if geom
                        else pyogrio.read_dataframe(GPKG, layer=layer, read_geometry=False))
    return _LAYERS[key].copy()


ASSERTIONS = []


def check(label, ok, detail=""):
    """Record and print an assertion. A figure must not disagree with the package."""
    ASSERTIONS.append((label, bool(ok), detail, CURRENT_ITEM))
    print(("  [OK]   " if ok else "  [FAIL] ") + label + (f" - {detail}" if detail else ""))
    if not ok:
        raise AssertionError(f"{label}: {detail}")


# ── cluster labels, approved: nothing below spells a cluster name out by hand ──
CL = S.labels()                                   # indexed by integer cluster number
CLUSTER_LABEL      = {int(i): r.label_short for i, r in CL.iterrows()}
CLUSTER_LABEL_LONG = {int(i): r.label_long for i, r in CL.iterrows()}


def cl_header(cid, n=None):
    """Panel header: cluster in words, its code, and how many cities it holds."""
    return S.panel_header(int(cid), n)


# ── output tree ──
OUT       = ROOT / "Results" / "manuscript_outputs"
MAIN_FIG  = OUT / "main_figures"
SUP_FIG   = OUT / "supplementary_figures"
TABLES    = OUT / "tables"
APPENDIX  = OUT / "appendices"
for d in (MAIN_FIG, SUP_FIG, TABLES, APPENDIX):
    d.mkdir(parents=True, exist_ok=True)


# ── data-source policy and provenance (plan Section 3) ───────────────────────
# Every file this notebook reads goes through read_source(): a forbidden source stops the
# run, and every read is logged (item, path, category, permitted, size, mtime, MD5) to
# _data_sources.csv. The v3.0 format references are opened only by compare_preview(),
# as pictures, never as data.
FORBIDDEN_PARTS = ("_superseded_", "_to_delete", "_archive_v1.0.0", "/notebooks/",  # forbidden-list
                   "stakeholder_hitrate_1000m")                                     # forbidden-list
PERMITTED = {          # cache or input -> the items allowed to read it (plan v3.2, Section 3)
    "input": {"Fig. 1", "S1", "Supp. Table S1"},
    "NB01": {"Fig. 3", "S1", "S2"},
    "NB03": {"Fig. 5", "Fig. 6", "S4"},
    "NB04": {"S5", "S7"},
    "NB05": {"Fig. 9", "Fig. 10", "S8"},
    "NB06": {"Table 4", "Table 5", "Fig. 15", "Supp. Table S2", "Supp. Table S8"},
}
SOURCES_CSV = OUT / "_data_sources.csv"
SOURCE_ROWS, _MD5 = [], {}
CURRENT_ITEM = "Setup"


def begin(item):
    """The item that the following reads and checks belong to."""
    global CURRENT_ITEM
    CURRENT_ITEM = item


def _category(p):
    p = Path(p).resolve()
    if p == GPKG.resolve():
        return "PKG"
    s = p.as_posix()
    for k in ("01", "02", "03", "04", "05", "06"):
        if f"/Results/{k}_" in s:
            return f"NB{k}"
    return "input" if "/data/inputs/" in s else "other"


def read_source(path, layer=""):
    """Refuse a forbidden source, log the read, and return the path unchanged."""
    p = Path(path)
    s = p.resolve().as_posix()
    if any(f in s for f in FORBIDDEN_PARTS):
        raise PermissionError(f"{CURRENT_ITEM}: forbidden data source {p}")
    cat = _category(p)
    permitted = ("yes" if cat == "PKG" or CURRENT_ITEM in PERMITTED.get(cat, ())
                 else "shared" if CURRENT_ITEM == "Setup" else "OOS")
    if p.is_file():
        if s not in _MD5:
            _MD5[s] = hashlib.md5(p.read_bytes()).hexdigest()
        size, h = p.stat().st_size, _MD5[s]
        mtime = pd.Timestamp(p.stat().st_mtime, unit="s").strftime("%Y-%m-%d %H:%M")
    else:
        size, mtime, h = (sum(1 for _ in p.iterdir()) if p.is_dir() else -1), "", \
            ("directory" if p.is_dir() else "missing")
    row = {"item": CURRENT_ITEM, "path": p.resolve().relative_to(ROOT.resolve()).as_posix()
           if p.resolve().is_relative_to(ROOT.resolve()) else s, "layer": layer,
           "category": cat, "permitted": permitted, "size": size, "mtime": mtime, "md5": h}
    SOURCE_ROWS.append(row)
    pd.DataFrame([row]).to_csv(SOURCES_CSV, mode="a", header=False, index=False)
    return path


# ── monitoring: side-by-side previews and print-size records ─────────────────
V3 = OUT / "_superseded_v3.0"          # visual format reference only (forbidden-list exempt)
PREVIEW = OUT / "_preview"
PREVIEW.mkdir(exist_ok=True)
FIG_REPORT = {}


def _as_page(p):
    """A written figure as a fitz page source: a real PDF, else its SVG."""
    import fitz as _fz
    p = Path(p)
    if p.suffix == ".pdf" and p.open("rb").read(5) == b"%PDF-":
        return _fz.open(str(p))
    svg = p.with_name(p.stem + ".svg") if p.suffix == ".pdf" else p
    return _fz.open(str(svg), filetype="svg")


def fig_report(fig, name):
    """Record the print size (mm) and smallest type (pt) of a figure or a written PDF."""
    if hasattr(fig, "get_size_inches"):
        import matplotlib.text as _mtext
        w, h = fig.get_size_inches() * 25.4
        sizes = [t.get_fontsize() for t in fig.findobj(_mtext.Text) if t.get_text().strip()]
    else:
        d = _as_page(fig)
        pg = d[0]
        w, h = pg.rect.width / 72 * 25.4, pg.rect.height / 72 * 25.4
        sizes = [sp["size"] for b in pg.get_text("dict")["blocks"] for ln in b.get("lines", [])
                 for sp in ln.get("spans", []) if sp.get("text", "").strip()]
        d.close()
    FIG_REPORT[name] = (round(w), round(h), round(min(sizes), 2) if sizes else None)
    return FIG_REPORT[name]


def compare_preview(item, v3_ref, new_out):
    """<item>_compare.png: the v3.0 reference (left) and the new output (right), same scale."""
    import io
    from PIL import Image, ImageDraw

    def render(p):
        if p is None or not Path(p).exists():
            img = Image.new("RGB", (500, 200), "white")
            ImageDraw.Draw(img).text((20, 90), "missing", fill=(160, 0, 0))
            return img, "missing"
        p = Path(p)
        if p.suffix == ".png":                      # a reference picture, e.g. a manuscript table
            img = Image.open(p).convert("RGB")
            return img, f"{img.width} x {img.height} px"
        if p.suffix in (".csv", ".xlsx"):
            import textwrap
            df = (pd.read_csv(p, dtype=str) if p.suffix == ".csv"
                  else pd.read_excel(p, dtype=str)).fillna("").iloc[:34, :12]
            # long cells wrap rather than being cut, so the preview reads like the printed table
            wid = [min(max([len(str(c))] + [len(v) for v in df[c]]), 40) for c in df.columns]
            wrap = lambda v, w: "\n".join(textwrap.wrap(str(v), w)) or ""
            head = [wrap(c, w) for c, w in zip(df.columns, wid)]
            cells = [[wrap(v, w) for v, w in zip(r, wid)] for r in df.values.tolist()]
            lines = [max(s.count("\n") + 1 for s in r) for r in [head] + cells]
            f = plt.figure(figsize=(0.07 * sum(wid), 0.15 * sum(lines)))
            ax = f.add_axes([0, 0, 1, 1]); ax.axis("off")
            t = ax.table(cellText=cells, colLabels=head, cellLoc="left", bbox=[0, 0, 1, 1],
                         colWidths=[w / sum(wid) for w in wid])
            t.auto_set_font_size(False); t.set_fontsize(6.5)
            for (i, _), c in t.get_celld().items():
                c.set_height(lines[i] / sum(lines))
            # through the Agg canvas: config.py (EXPORT_MODE "svg_only") patches Figure.savefig
            # so that a PNG request is written as SVG
            from matplotlib.backends.backend_agg import FigureCanvasAgg
            buf = io.BytesIO()
            FigureCanvasAgg(f).print_figure(buf, format="png", dpi=90, bbox_inches="tight")
            plt.close(f)
            return Image.open(io.BytesIO(buf.getvalue())).convert("RGB"), \
                f"{len(df)} rows x {len(df.columns)} cols"
        d = _as_page(p)
        pg = d[0]
        pm = pg.get_pixmap(dpi=90, alpha=False)
        info = f"{pg.rect.width / 72 * 25.4:.0f} x {pg.rect.height / 72 * 25.4:.0f} mm"
        d.close()
        return Image.frombytes("RGB", (pm.width, pm.height), pm.samples), info

    L, li = render(v3_ref)
    R, ri = render(new_out)
    img = Image.new("RGB", (L.width + R.width + 30, max(L.height, R.height) + 30), "white")
    d = ImageDraw.Draw(img)
    d.text((6, 8), f"{item}  v3.0: {Path(v3_ref).name if v3_ref else '-'}  [{li}]", fill="black")
    d.text((L.width + 30, 8), f"new: {Path(new_out).name if new_out else '-'}  [{ri}]",
           fill="black")
    img.paste(L, (0, 30)); img.paste(R, (L.width + 30, 30))
    d.line([(L.width + 15, 0), (L.width + 15, img.height)], fill=(120, 120, 120), width=2)
    m = re.match(r"(Fig\.|Table) (\d+)$", item)
    tag = ((f"Fig_{int(m.group(2)):02d}" if m.group(1) == "Fig." else f"Table_{m.group(2)}") if m
           else item.replace("Supp. Table ", "Supp_Table_").replace(" ", "_"))
    out = PREVIEW / f"{tag}_compare.png"
    img.save(out)
    return out


# ── NB05 artwork collection (Figs 8-9 and the supplementary sets read NB05 figures) ──
RESULTS_06T = RESULTS_06 / "tables"
_F05 = RESULTS_05 / "figures"
_REPAIRED = set()


def _copy_pair(stem_src, dst_dir, stem_dst):
    """Collect an NB05 figure pair, repairing the extension where it lies.

    133 of NB05's 136 .pdf figure artefacts hold SVG content: the saves there relied on
    the extension to pick the format and matplotlib wrote SVG into both files. Rather than
    ship a .pdf no PDF reader can open, the content is checked here and a real PDF is
    rendered from the SVG. NB05 itself is left untouched.
    """
    import fitz as _fz
    picked = []
    for ext in ("svg", "pdf"):
        s = _F05 / f"{stem_src}.{ext}"
        if not s.exists():
            continue
        read_source(s)
        d = dst_dir / f"{stem_dst}.{ext}"
        head = s.open("rb").read(5)
        if ext == "pdf" and head != b"%PDF-":
            _svg = _F05 / f"{stem_src}.svg"
            _src = _svg if _svg.exists() else s
            d.write_bytes(_fz.open(str(_src)).convert_to_pdf())
            _REPAIRED.add(stem_src)
        else:
            shutil.copyfile(s, d)
        picked.append(ext)
    return picked


SOURCES_CSV.write_text("item,path,layer,category,permitted,size,mtime,md5\n", encoding="utf-8")

# ── print geometry: every figure is built at its final size on the page ──
MM = 1.0 / 25.4                      # mm -> inches
W_FULL, W_1_5, W_1COL = 190 * MM, 140 * MM, 90 * MM     # 190 / 140 / 90 mm
H_MAX = 240 * MM                                         # no taller than 240 mm
MIN_PT = 6.5                                             # no text smaller than this


def fig_size_report(paths=None):
    """For every main figure: its size in mm and the smallest type on the page.

    Text is measured from the written PDF, so it reports what the printer sees,
    including figures composed from other PDFs.
    """
    import fitz as _fzr
    rows = []
    for p in sorted(paths or MAIN_FIG.glob("Fig_*.pdf")):
        d = _fzr.open(p)
        pg = d[0]
        w_mm, h_mm = pg.rect.width / 72.0 * 25.4, pg.rect.height / 72.0 * 25.4
        sizes = [sp["size"] for bl in pg.get_text("dict")["blocks"]
                 for ln in bl.get("lines", []) for sp in ln.get("spans", [])
                 if sp.get("text", "").strip()]
        d.close()
        rows.append({"figure": p.stem, "width_mm": round(w_mm, 1),
                     "height_mm": round(h_mm, 1),
                     "min_pt": round(min(sizes), 2) if sizes else np.nan,
                     "n_text": len(sizes)})
    r = pd.DataFrame(rows)
    if len(r):
        r["width_ok"] = r.width_mm <= 190.5
        r["height_ok"] = r.height_mm <= 240.5
        r["font_ok"] = r.min_pt.isna() | (r.min_pt >= MIN_PT - 0.05)
        r["ok"] = r.width_ok & r.height_ok & r.font_ok
    return r


PAD_IN = 0.02                 # the margin savefig adds on each side


def _raise_small_type(fig, min_pt=None):
    """Lift every text artist below the minimum point size.

    Sweeping the source for `fontsize=` missed `labelsize=`, `set_fontsize` and the
    rcParam defaults, so the rule is applied to the drawn artists instead, where
    nothing can hide. Returns the smallest size found before the lift.
    """
    import matplotlib.text as _mtext
    m = min_pt or MIN_PT
    smallest = None
    for t in fig.findobj(_mtext.Text):
        if not t.get_text().strip():
            continue
        sz = t.get_fontsize()
        smallest = sz if smallest is None else min(smallest, sz)
        if sz < m:
            t.set_fontsize(m)
    return smallest


def _fit_width(fig, target_in, tries=5):
    """Shrink the figure until the SAVED page is the target print width.

    bbox_inches="tight" grows the page past figsize whenever a legend or label sits
    outside the axes, and then adds pad_inches on each side, so figsize alone never
    controlled the printed width. Text keeps its point size while the canvas shrinks,
    so this converges without making type smaller.
    """
    inner = target_in - 2 * PAD_IN
    for _ in range(tries):
        fig.canvas.draw()
        bb = fig.get_tightbbox(fig.canvas.get_renderer())
        if bb is None or bb.width <= 0:
            return
        k = inner / bb.width
        if abs(k - 1.0) < 0.002:
            return
        w, h = fig.get_size_inches()
        fig.set_size_inches(w * k, h, forward=True)


def _page_width_mm(p):
    import fitz as _fz
    d = _fz.open(p)
    w = d[0].rect.width / 72.0 * 25.4
    d.close()
    return w


def _scale_pdf_page(p, target_in):
    """Rescale a written PDF page to the target width, keeping its proportions."""
    import fitz as _fz
    src = _fz.open(p)
    sw, sh = src[0].rect.width, src[0].rect.height
    k = (target_in * 72.0) / sw
    out = _fz.open()
    pg = out.new_page(width=sw * k, height=sh * k)
    pg.show_pdf_page(pg.rect, src, 0)
    out.save(str(p) + ".tmp")
    src.close()
    svg = _fz.open(str(p) + ".tmp")[0].get_svg_image()
    out.close()
    Path(str(p) + ".tmp").replace(p)
    return svg


def show_file(paths):
    """Display a written figure (its PDF) with its print size and smallest type."""
    from IPython.display import Image as _Image
    pdf = next(p for p in paths if p.suffix == ".pdf")
    w, h, pt = fig_report(pdf, CURRENT_ITEM)
    d = _as_page(pdf)
    png = d[0].get_pixmap(dpi=120, alpha=False).tobytes("png")
    d.close()
    print(f"Saved: {', '.join(p.name for p in paths)} in {pdf.parent.relative_to(ROOT)}  ·  "
          f"{w} × {h} mm, " + (f"smallest type {pt} pt" if pt else "text drawn as paths"))
    display(_Image(data=png))


def savefig(fig, path_noext, formats=("pdf", "svg"), dpi=300, fit=W_FULL, as_v30=False,
            show=False):
    """Save one figure to vector formats and return the paths written.

    format= is passed explicitly and the written magic bytes are checked: relying on
    the extension alone once produced an SVG inside a .pdf, which no visual check
    would catch.

    Fitting to `fit` inches of print width is tried first by shrinking the canvas,
    which leaves type at its point size. Some figures hold an artist whose extent is
    fixed in inches, so the canvas cannot shrink past it; those are drawn with type
    enlarged by the overshoot and the finished page is then scaled down, which lands
    the printed type back on MIN_PT rather than below it.

    as_v30=True saves as v3.0's savefig did, for items restored to their v3.0 format: tight
    bounding box with matplotlib's default 0.1 in padding, no type lifting, no fitting to a
    print width (plan 1a.5: sizes are reported, never forced).

    show=True closes the live figure and displays the written PDF instead, with its print size
    and smallest type, so what the notebook shows is exactly the file in the output folder.
    """
    if as_v30:
        fit = None
    else:
        _raise_small_type(fig)
    if fit:
        _fit_width(fig, fit)
        _raise_small_type(fig)
    out = []
    for ext in formats:
        p = Path(f"{path_noext}.{ext}")
        fig.savefig(p, format=ext, dpi=dpi, bbox_inches="tight",
                    pad_inches=0.1 if as_v30 else PAD_IN)
        out.append(p)

    if fit:
        over = _page_width_mm(out[0]) / (fit * 25.4)
        if over > 1.004:
            _raise_small_type(fig, MIN_PT * over)
            _fit_width(fig, fit)
            for ext, p in zip(formats, out):
                fig.savefig(p, format=ext, dpi=dpi, bbox_inches="tight",
                            pad_inches=PAD_IN)
            svg = _scale_pdf_page(out[0], fit)
            for p in out[1:]:
                if p.suffix == ".svg":
                    if not svg.lstrip().startswith("<?xml"):
                        svg = '<?xml version="1.0" encoding="utf-8" standalone="no"?>'                               + chr(10) + svg
                    p.write_text(svg, encoding="utf-8")

    for p in out:
        head = p.open("rb").read(5)
        want = b"%PDF-" if p.suffix == ".pdf" else (b"<?xml" if p.suffix == ".svg" else None)
        if want and head != want:
            raise RuntimeError(f"{p.name} was written as {head!r}, not {p.suffix}")

    if show:
        plt.close(fig)
        show_file(out)
    return out


print("Repo root :", ROOT)
print("Package   :", GPKG.relative_to(ROOT))
print("Output    :", OUT.relative_to(ROOT))
print("Clusters  :", len(CL), "labelled,", int(CL.has_study_cities.sum()), "with study cities")
print("Sources   :", SOURCES_CSV.relative_to(ROOT), "(every read is logged here)")

In [ ]:
# 1) per-city zone GeoDataFrames -------------------------------------------------
zone_results = {}
for d in sorted(p for p in RESULTS_AOI.iterdir() if p.is_dir()):
    s = d.name
    cr  = d / f"{s}_city_region_4326_v5.geojson"
    uc  = d / f"{s}_urban_core_4326_v5.geojson"
    bra = d / f"{s}_bra_4326_v5.geojson"
    if cr.exists() and uc.exists() and bra.exists():
        zone_results[s.replace("_", " ")] = {
            "city_slug": s, "city_dir": d,
            "cr_gdf": gpd.read_file(read_source(cr)), "uc_gdf": gpd.read_file(read_source(uc)),
            "bra_gdf": gpd.read_file(read_source(bra))}
STUDY_CITIES = sorted(zone_results)          # 56 non-absorbed AOIs

# 2) WUP city population table (all candidates) ---------------------------------
wup = pd.read_csv(read_source(ROOT / config.AOI_DATA_PATHS["cities_csv"]))
YEAR_COLS = [c for c in wup.columns if str(c).isdigit()]
for c in YEAR_COLS:
    wup[c] = wup[c].astype(str).str.replace(",", "", regex=False).astype(float)  # thousands
wup_pts = gpd.GeoDataFrame(
    wup, geometry=[Point(xy) for xy in zip(wup["PWCent_Longitude"], wup["PWCent_Latitude"])],
    crs="EPSG:4326")

# 3) WUP cities inside each AOI (for composite / per-AOI trends) -----------------
cities_in_aoi = {}
for city, res in zone_results.items():
    crg = res["cr_gdf"].to_crs("EPSG:4326")
    inside = gpd.sjoin(wup_pts, crg, predicate="within", how="inner")
    inside = inside.drop(columns=[c for c in inside.columns if c.startswith("index_")], errors="ignore")
    cities_in_aoi[city] = {"cities_in_aoi_df": pd.DataFrame(inside.drop(columns="geometry"))}

# 4) authoritative per-AOI population -------------------------------------------
zpop = pd.read_csv(read_source(RESULTS_AOI / "zone_area_summary_with_pop.csv"))
pop_override = {str(r["city"]).strip(): float(r["population_wup_k"]) * 1000
               for _, r in zpop.iterrows() if pd.notna(r["population_wup_k"])}

# 5) active Cluster25 IDs --------------------------------------------------------
lookup = pd.read_csv(read_source(RESULTS_AOI / "city_cluster25_lookup.csv"))
active_clusters = sorted(int(x) for x in lookup["Cluster25"].dropna().unique())

# 6) landscape polygons + reference palette -------------------------------------
landscape_gdf = gpd.read_file(read_source(config.LANDSCAPE_DATA_PATH))
palette = pd.read_csv(read_source(RESULTS_AOI / "cluster25_reference_palette.csv"))

print(f"AOIs (zone_results)      : {len(zone_results)}")
print(f"WUP candidate cities     : {len(wup)}")
print(f"Active Cluster25 IDs (16): {active_clusters}")
print(f"Landscape polygons       : {len(landscape_gdf)}   Palette rows: {len(palette)}")

## 1 · Main article

### Front matter

#### Graphical abstract · AUTHOR

The author produces it last, from the final numbers. No code here.

### 1.1 · Methods (manuscript section 2)

#### Fig. 1 · Study area

**Manuscript** 2.1 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_1_study_area.svg` (208 × 188 mm) · **Code** the cell below, restored from the v3.0 cell (`Notebook/publication_outputs.ipynb` @ `d2fef8e`) · **Output** `main_figures/Fig_01_study_area`

**Data** package `city_profile` (cities, clusters, `population_wup_k`, `absorbed_cities`); inputs: UN WUP table (positions; the 11 absorbed cities' populations), typology map, national boundary

**Changes from v3.0** (the author's, 30 Oct): no title; all 67 candidate cities with the one city symbol (the 11 absorbed ones drawn, not named); the typology legend under the map, all 24 classes as climate · geology · elevation with C-codes; the population key (300,000 / 1 million / 3 million / 10 million) inside the map at the lower right. The cell shows the saved PDF.

In [ ]:
begin("Fig. 1")
# ── Fig. 1 · Study area (v3.0 Figure 1; cell restored from publication_outputs.ipynb @ d2fef8e) ──
# The v3.0 call, unchanged: A4 portrait, the same map body, the 56 study cities named with
# their own UN WUP 2024 population, saved as v3.0 saved it; the notebook shows the saved
# PDF. Changed as the author asked:
#   - no in-figure title;
#   - all 67 candidate cities with the one city symbol; the 11 absorbed cities are drawn
#     but not named, so the names on the map are v3.0's;
#   - the typology legend under the map body: all 24 classes, as climate · geology ·
#     elevation with codes C1 ... C25;
#   - the population key (300,000 / 1 million / 3 million / 10 million) inside the map,
#     at the lower right;
#   - city names, clusters and populations from the package, not the NB01 caches. Positions,
#     and the absorbed cities' populations, come from the UN WUP table (a permitted input).
_prof1 = pkg("city_profile")
_wup1 = pd.read_csv(read_source(ROOT / config.AOI_DATA_PATHS["cities_csv"]), thousands=",")
_wup1 = _wup1.assign(City_Name=_wup1.City_Name.str.strip()).set_index("City_Name")
_absorbed1 = sorted({n.strip() for s in _prof1.absorbed_cities.dropna() for n in str(s).split(";")})
_clusters1 = sorted(int(c) for c in _prof1.cluster25.unique())
_pop1 = {**dict(zip(_prof1.city, _prof1.population_wup_k * 1000)),
         **(_wup1.loc[_absorbed1, str(STUDY_YEAR)].astype(float) * 1000).to_dict()}
check("Fig. 1 - 67 candidate cities: 56 study cities and 11 absorbed ones, all in the WUP table",
      len(_prof1) == 56 and len(_absorbed1) == 11 and len(_pop1) == 67
      and set(_pop1) <= set(_wup1.index), f"{len(_prof1)} + {len(_absorbed1)} = {len(_pop1)}")
check("Fig. 1 - study-city populations: city_profile.population_wup_k equals the WUP 2024 input",
      np.array_equal(_prof1.population_wup_k.astype(float).values,
                     _wup1.loc[_prof1.city, str(STUDY_YEAR)].astype(float).values),
      f"56 of 56; Lagos {_pop1['Lagos'] / 1e6:.1f} million, Onitsha {_pop1['Onitsha'] / 1e6:.1f} million")
check("Fig. 1 - every candidate city is at or above the 300,000 threshold",
      min(_pop1.values()) >= 300_000, f"smallest {min(_pop1.values()):,.0f} ({min(_pop1, key=_pop1.get)})")
check("Fig. 1 - the legend lists all 24 typology classes with their approved labels",
      len(CL) == 24 and set(_clusters1) <= set(CL.index), f"{len(CL)} classes, {len(_clusters1)} with study cities")

fig1 = viz.create_publication_nigeria_overview(
    aoi_dict=dict.fromkeys(_prof1.city),           # the function uses only the city names
    cities_csv_path=read_source(ROOT / config.AOI_DATA_PATHS["cities_csv"]),
    landscape_path=read_source(config.LANDSCAPE_DATA_PATH),
    nigeria_path=read_source(config.GHSL_POP_CONFIG["boundary_geojson"]),
    cluster25_colors=config.CLUSTER25_COLORS,
    active_cluster_ids=_clusters1,
    study_year=STUDY_YEAR,
    pop_override=_pop1,
    save_path=None, dpi=300, figsize=(8.27, 11.69),
    # the author's changes
    title=None,
    unlabelled_cities=_absorbed1,
    city_legend_label="City (UN WUP 2024, ≥ 300,000 inhabitants)",
    pop_legend_values=[(10_000_000, "10 million"), (3_000_000, "3 million"),
                       (1_000_000, "1 million"), (300_000, "300,000")],
    legend_cluster_ids=sorted(CL.index),
    cluster_labels={i: f"{CL.loc[i].cluster25}  {CL.loc[i].label_short}" for i in CL.index},
    cluster_legend_heading="Landscape typology (climate · geology · elevation):",
    legend_below=True, legend_ncol=3)

paths = savefig(fig1, MAIN_FIG / "Fig_01_study_area", as_v30=True, show=True)

#### Fig. 2 · Methodological workflow · Drawn using Draw.io



#### Fig. 3 · Delineation excerpt (Kano, Abuja, Jos, Lagos, Port Harcourt)

**Manuscript** 2.3 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_4_city_aoi_grid` · **Code** the cell below

**Data** NB01 zone, water and road layers (permitted); typology map

In [ ]:
begin("Fig. 3")
# Unchanged definition; only the output name follows the new numbering.
GRID_CITIES = ["Kano", "Abuja", "Jos", "Lagos", "Port Harcourt"]
fig4 = viz.create_publication_city_grid(
    city_names=GRID_CITIES, zone_results=zone_results, results_aoi_path=read_source(RESULTS_AOI),
    landscape_gdf=landscape_gdf, cluster25_colors=config.CLUSTER25_COLORS,
    storage_crs="EPSG:4326", save_path=None, dpi=300, figsize=(8.27, 5.845),
    single_north_arrow=True)

paths = savefig(fig4, MAIN_FIG / "Fig_03_delineation_excerpt")
plt.show()
print("Saved:", [p.name for p in paths])

#### Table 1 · Typology composition of the 56 cities

**Manuscript** 2.3 · **v3.0 reference** the v3.0 manuscript Table 2 (`docs/scs_rebuild/reference/Table1_v3_manuscript.png`) · **Code** the cell below, restored from the v3.0 cell (`Notebook/publication_outputs.ipynb` @ `d2fef8e`, cell 17) · **Output** `tables/Table_01_typology_composition` (.csv, .xlsx)

**Data** package `city_profile` (cluster25, city, `population_wup_merged_k`, `cr_area_km2`, `uc_area_km2`); approved labels (`cluster_labels.csv`)

**Changes from v3.0** (plan 1a): values from the package, not the NB01 summaries; the manuscript layout (seven columns; eleven cluster rows, then Other with five single-city clusters); descriptions from the approved labels; whole numbers with thousands separators; C13 footnote in the workbook.

In [ ]:
begin("Table 1")
# ── Table 1 · Typology composition (v3.0 manuscript Table 2; cell restored from
#    publication_outputs.ipynb @ d2fef8e, cell 17) ──
# The v3.0 export, unchanged: a CSV and a formatted workbook. The layout is the v3.0
# manuscript table (docs/scs_rebuild/reference/Table1_v3_manuscript.png). Changed as plan
# Section 1a allows:
#   - values from city_profile (the package), not the NB01 summaries;
#   - the manuscript's seven columns; clusters by number of cities, then the five
#     single-city clusters grouped as "Other", each city named with its cluster;
#   - descriptions from the approved labels (climate · geology · elevation, with range);
#   - whole numbers with thousands separators; the C13 footnote in the workbook.
_prof1 = pkg("city_profile")
OTHER_CLUSTERS = [6, 8, 15, 21, 23]                  # grouped as in the v3.0 manuscript
_g1 = _prof1.assign(cid=_prof1.cluster25.astype(int)).groupby("cid")
_agg1 = pd.DataFrame({"n": _g1.city.size(), "cities": _g1.city.agg(lambda s: ", ".join(sorted(s))),
                      "pop": _g1.population_wup_merged_k.sum(),
                      "cr": _g1.cr_area_km2.sum(), "uc": _g1.uc_area_km2.sum()})
_multi1 = _agg1.drop(OTHER_CLUSTERS).sort_values("n", ascending=False, kind="stable")
_oth1 = _agg1.loc[OTHER_CLUSTERS]
_num1 = ["WUP pop. (000s)", "CR (km²)", "UC (km²)"]
tab1 = pd.DataFrame(
    [[f"C{c}", int(r["n"]), r["cities"], f"{CL.loc[c].label_short} ({CL.loc[c].elevation_range})",
      float(r["pop"]), float(r["cr"]), float(r["uc"])] for c, r in _multi1.iterrows()]
    + [[f"Other ({len(_oth1)} clusters)", int(_oth1.n.sum()),
        ", ".join(f"{r['cities']} (C{c})" for c, r in _oth1.iterrows()), "Mixed",
        float(_oth1["pop"].sum()), float(_oth1.cr.sum()), float(_oth1.uc.sum())]],
    columns=["Cluster Group", "No. of cities", "Cities", "Physiographic description"] + _num1)

check("Table 1 - rows as in the v3.0 manuscript: 11 clusters by number of cities, then Other",
      list(tab1["Cluster Group"]) == ["C13", "C4", "C9", "C7", "C25", "C11", "C1", "C12", "C16",
                                      "C17", "C19", "Other (5 clusters)"],
      ", ".join(f"{g} {n}" for g, n in zip(tab1["Cluster Group"], tab1["No. of cities"])))
check("Table 1 - the 56 study cities, each in exactly one row",
      tab1["No. of cities"].sum() == 56 and _agg1.n.sum() == _prof1.city.nunique() == 56,
      f"{tab1['No. of cities'].sum()} cities in {len(tab1)} rows")
check("Table 1 - population, CR and UC columns sum to the city_profile totals",
      np.allclose(tab1[_num1].sum().values, [_prof1.population_wup_merged_k.sum(),
                                             _prof1.cr_area_km2.sum(), _prof1.uc_area_km2.sum()]),
      f"{tab1[_num1[0]].sum():,.0f} thousand; {tab1[_num1[1]].sum():,.0f} km² CR; "
      f"{tab1[_num1[2]].sum():,.0f} km² UC")

_uc13, _uct1 = _agg1.loc[13, "uc"], _prof1.uc_area_km2.sum()
FOOT_T1 = (f"C13 alone holds {100 * _uc13 / _uct1:.1f}% of the urban-core (UC) area of the "
           f"{len(_prof1)} study cities ({_uc13:,.0f} of {_uct1:,.0f} km²).")

csv1 = TABLES / "Table_01_typology_composition.csv"
tab1.assign(**{c: tab1[c].map("{:,.0f}".format) for c in _num1}).to_csv(
    csv1, index=False, encoding="utf-8-sig")

xls1 = TABLES / "Table_01_typology_composition.xlsx"
wb = xlsxwriter.Workbook(str(xls1)); ws = wb.add_worksheet("Table 1")
hdr  = wb.add_format({"bold": True, "bg_color": "#404040", "font_color": "white",
                      "border": 1, "align": "center", "valign": "vcenter", "text_wrap": True})
cell = wb.add_format({"border": 1, "valign": "vcenter", "text_wrap": True})
num  = wb.add_format({"border": 1, "valign": "vcenter", "num_format": "#,##0"})
foot = wb.add_format({"italic": True, "font_size": 8})
for j, c in enumerate(tab1.columns):
    ws.write(0, j, c, hdr)
for i, r in enumerate(tab1.itertuples(index=False), start=1):
    for j, (c, v) in enumerate(zip(tab1.columns, r)):
        ws.write(i, j, v, num if c in _num1 else cell)
ws.write(len(tab1) + 2, 0, FOOT_T1, foot)
ws.set_column(0, 0, 14); ws.set_column(1, 1, 9); ws.set_column(2, 2, 46)
ws.set_column(3, 3, 44); ws.set_column(4, 6, 11)
wb.close()

print(f"Saved: {csv1.name}, {xls1.name} in {TABLES.relative_to(ROOT)}")
display(pd.read_csv(csv1, dtype=str, encoding="utf-8-sig"))
print(FOOT_T1)

#### Fig. 4 · Stakeholder participants by cluster, city and participation (rings)

**Manuscript** 2.3 (stakeholder survey) · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_5_stakeholder_radial` (the NB02 §6 chart) · **Code** the cell below, restored from the v3.0 cell (`Notebook/publication_outputs.ipynb` @ `d2fef8e`, cell 25) · **Output** `main_figures/Fig_04_stakeholder_rings`

**Data** package `city_profile` (56 cities, clusters) and `stakeholder_perception` (participants per city)

**Changes from v3.0** (plan 1a; moved from the Supplementary, author 30 Oct): inputs from the package; no title; legend "att." written out.

In [ ]:
begin("Fig. 4")
# ── Fig. 4 · Stakeholder rings (v3.0 Figure 5; cell restored from publication_outputs.ipynb
#    @ d2fef8e, cell 25; the NB02 §6 chart, moved into the main text, author 30 Oct) ──
# The v3.0 call, unchanged: create_city_stakeholder_donut at figsize (16, 9) -- ring 1 the 16
# clusters, ring 2 the attendance split, ring 3 the 56 cities, the 21 cities with participants
# labelled -- saved as v3.0 saved it. Changed only as the plan allows:
#   - its inputs built from the package (city_profile, stakeholder_perception) instead of the
#     NB02 caches, with the same columns; the rings use participants, never registrations;
#   - no in-figure title; the legend's "att." written out.
_sp4 = pkg("stakeholder_perception")
_cp4 = pkg("city_profile")
_study4 = pd.DataFrame({"city": _cp4.city, "Cluster25": _cp4.cluster25.astype(int),
                        "total_attendees": _cp4.city.map(_sp4.groupby("city").size())
                                           .fillna(0).astype(int)})
_clus4 = (_study4.groupby("Cluster25")
          .agg(**{"N Cities": ("city", "size"), "Attended Workshop": ("total_attendees", "sum")})
          .reset_index().assign(Cluster=lambda d: "C" + d.Cluster25.astype(str)))
_with4 = _study4[_study4.total_attendees > 0]
check("Fig. 4 - the perception cohort: 105 participants, 21 cities, 10 clusters",
      (len(_sp4), _sp4.participant_code.nunique(), _with4.city.nunique(),
       _with4.Cluster25.nunique()) == (105, 105, 21, 10),
      f"{_study4.total_attendees.sum()} participants, {len(_with4)} cities, "
      f"{_with4.Cluster25.nunique()} clusters")
check("Fig. 4 - every participant's city and cluster are in city_profile",
      set(_sp4.city) <= set(_cp4.city)
      and (_sp4.cluster25.astype(int).values
           == _cp4.set_index("city").cluster25.astype(int).reindex(_sp4.city).values).all(),
      "105 of 105")
check("Fig. 4 - the rings hold the 56 study cities in 16 clusters",
      _clus4["N Cities"].sum() == 56 and len(_clus4) == 16
      and _clus4["Attended Workshop"].sum() == len(_sp4),
      ", ".join(f"{r.Cluster} {r['N Cities']}/{r['Attended Workshop']}" for _, r in _clus4.iterrows()))

fig4 = viz.create_city_stakeholder_donut(
    cluster_summary=_clus4[["Cluster", "N Cities", "Attended Workshop"]], study_gdf=_study4,
    cluster25_colors=config.CLUSTER25_COLORS, attended_pids=len(_sp4), figsize=(16, 9),
    show_title=False, legend_count_label="participants")
paths = savefig(fig4, MAIN_FIG / "Fig_04_stakeholder_rings", as_v30=True, show=True)

### 1.2 · Results 3.1 — long-term and event-scale surface temperature (Objective 1)

#### Fig. 5 · Annual trends, selected clusters

**Manuscript** 3.1 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_7_annual_trend` · **Code** the cell below, restored from the v3.0 cells (`Notebook/publication_outputs.ipynb` @ `d2fef8e`, cells 30 + 32); panels by `src/temporal_panels.py`, the annual half of NB03's `create_temporal_chart` · **Output** `main_figures/Fig_05_annual_trends`

**Data** NB03 cluster composite series (permitted), asserted against package `annual_series`; trend table from package `composite_trend_statistics`; panel headers from `cluster_labels.csv`

**Changes from v3.0** (plan 1a): table values from the package; output name. The C1 merge with the monthly figure is undone.

In [ ]:
begin("Fig. 5")
# ── Fig. 5 · Annual trends (v3.0 Figure 7; cells restored from publication_outputs.ipynb
#    @ d2fef8e, cells 30 + 32) ──
# The v3.0 definition, unchanged: render_cluster_grid() draws each cluster with the NB03
# temporal panel (src/temporal_panels.py, the annual half of NB03's create_temporal_chart)
# on an A4 page with the legend and the trend-summary table below, saved as v3.0 saved it.
# Fig. 6 calls the same renderer for the monthly half. Changed only as the plan allows:
#   - the panel headers come from the approved label file (same text as v3.0's NB01 read);
#   - Fig. 5's table from the package (composite_trend_statistics); the series drawn are
#     asserted against the package;
#   - output names; the notebook shows the saved file.
import matplotlib.patches as _mpat
from matplotlib.lines import Line2D as _L2D
import temporal_panels as tp

STATS03    = RESULTS_03 / "Cluster25_Composites" / "stats"

SELECTED_CLUSTERS = [4, 7, 9, 13, 19, 25]
CL_META = {int(i): (r.climate, int(r.n_cities)) for i, r in CL.iterrows()}

def _temporal_legend(fig, y):
    Cc = tp.NB03_COLOURS
    fig.legend(handles=[
        _L2D([0],[0], color=Cc['uc_day'],    lw=1.4, label='UC — day'),
        _L2D([0],[0], color=Cc['uc_night'],  lw=1.4, label='UC — night'),
        _L2D([0],[0], color=Cc['bra_day'],   lw=1.4, label='BRA — day'),
        _L2D([0],[0], color=Cc['bra_night'], lw=1.4, label='BRA — night'),
        _L2D([0],[0], color=Cc['delta_day'], lw=1.8, label='SUHII — day'),
        _L2D([0],[0], color=Cc['delta_night'],lw=1.8, label='SUHII — night'),
        _mpat.Patch(facecolor=Cc['cr_day_fill'],   alpha=.55, label='CR range — day'),
        _mpat.Patch(facecolor=Cc['cr_night_fill'], alpha=.55, label='CR range — night'),
        _L2D([0],[0], color='#777', lw=.9, ls='--', label="Sen's slope"),
    ], loc='center', ncol=5, fontsize=7, framealpha=.9, bbox_to_anchor=(0.5, y))

def _cluster_title(cid):
    clim, n = CL_META.get(cid, ("", 0)); return f"C{cid} — {clim} ({n} cities)"

def _trend_cell(stats, dstats, zone, period):
    src = dstats if zone == "delta_uc_bra" else stats
    if src is None or src.empty:
        return "—"
    r = src[(src["zone"] == zone) & (src["period"] == period)]
    if r.empty:
        return "—"
    r = r.iloc[0]
    sym = {"increasing": "↑", "decreasing": "↓"}.get(str(r.get("mk_trend", "")), "—")
    try:
        pf = float(r.get("mk_p")); sig = "***" if pf < .001 else ("**" if pf < .01 else ("*" if pf < .05 else "ns"))
    except (TypeError, ValueError):
        sig = ""
    try:
        dec = f"{float(r.get('sen_slope_per_decade')):+.2f}"
    except (TypeError, ValueError):
        dec = ""
    hr = "ᴴ" if r.get("autocorr_corrected", False) else ""
    return f"{sym}{sig} {dec}{hr}".strip()

def render_cluster_grid(cids, kind, save_noext, trend_cell=None):
    """2-col × 3-row cluster grid (no in-panel stat boxes) + a summary table below the legend.

    trend_cell(cid, zone, period), when given, fills the table instead of the NB03 stats."""
    fig = plt.figure(figsize=(8.27, 11.69))
    gs = fig.add_gridspec(3, 2, top=0.965, bottom=0.345, left=0.10, right=0.90,
                          hspace=0.55, wspace=0.50)
    trows = []
    for k, cid in enumerate(cids):
        ax = fig.add_subplot(gs[k // 2, k % 2]); ax2 = ax.twinx()
        df, ddf, st, ds = tp.panel_inputs(tp.load_cluster(cid, read_source(STATS03)), kind)
        tp.draw_temporal_panel(ax, ax2, kind, df, ddf, st, ds, stat_boxes=False)
        ax.set_title(_cluster_title(cid), fontsize=8, fontweight="bold", pad=3)
        cell = ((lambda z, p: trend_cell(cid, z, p)) if trend_cell
                else (lambda z, p: _trend_cell(st, ds, z, p)))
        trows.append([f"C{cid}",
                      cell("urban_core", "day"),  cell("bra", "day"),  cell("delta_uc_bra", "day"),
                      cell("urban_core", "night"), cell("bra", "night"), cell("delta_uc_bra", "night")])
    _temporal_legend(fig, y=0.30)
    # ── in-panel summary table (holds the per-cluster trend detail; not referenced elsewhere) ──
    tax = fig.add_axes([0.05, 0.035, 0.90, 0.215]); tax.axis("off")
    cols = ["Cluster", "UC · Day", "BRA · Day", "SUHII · Day", "UC · Night", "BRA · Night", "SUHII · Night"]
    tbl = tax.table(cellText=trows, colLabels=cols, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(6.4); tbl.scale(1, 1.55)
    for (rr, cc), cl in tbl.get_celld().items():
        cl.set_edgecolor("#cccccc")
        if rr == 0:
            cl.set_facecolor("#404040"); cl.set_text_props(color="white", fontweight="bold")
        elif cc == 0:
            cl.set_text_props(fontweight="bold")
    tax.set_title("Trend summary — ↑/↓ Mann-Kendall direction · */**/*** = p<.05/.01/.001 · "
                  "value = Sen slope (°C/decade) · ᴴ = Hamed–Rao corrected",
                  fontsize=6.5, pad=4)
    return savefig(fig, save_noext, as_v30=True, show=True)


# ── SCS rebuild: the package behind the table and the panels ─────────────────────
# Fig. 5's table is filled from the published composite_trend_statistics, after the NB03
# stats it replaces are asserted to agree. Zone names differ between the two: tp uses
# urban_core / bra / delta_uc_bra, the package uses uc / bra / suhii.
_ZONE_PKG = {"urban_core": "uc", "bra": "bra", "delta_uc_bra": "suhii"}
_COMP = pkg("composite_trend_statistics")
_COMP = _COMP[_COMP.composite_type == "cluster"]


def _pkg_trend(cid, zone, period):
    """One row of the published cluster composite, by tp zone name."""
    r = _COMP[(_COMP.composite_key == f"C{int(cid)}")
              & (_COMP.zone == _ZONE_PKG[zone]) & (_COMP.period == period)]
    return None if r.empty else r.iloc[0]


def _trend_cell_pkg(cid, zone, period):
    """Same cell format as _trend_cell, but sourced from the package."""
    r = _pkg_trend(cid, zone, period)
    if r is None:
        return "—"
    sym = {"increasing": "↑", "decreasing": "↓"}.get(str(r.mk_trend), "—")
    p = float(r.mk_p_value)
    sig = "***" if p < .001 else ("**" if p < .01 else ("*" if p < .05 else "ns"))
    hr = "ᴴ" if bool(r.autocorr_corrected) else ""
    return f"{sym}{sig} {float(r.sen_slope_c_per_decade):+.2f}{hr}".strip()


def assert_trends_match_package(cids, tol=1e-5):
    """The NB03 stats behind the panels and the published composites must agree."""
    worst, n = 0.0, 0
    for cid in cids:
        ent = tp.load_cluster(cid, read_source(STATS03))
        _, _, st, ds = tp.panel_inputs(ent, "annual")
        for zone in ("urban_core", "bra", "delta_uc_bra"):
            src = ds if zone == "delta_uc_bra" else st
            if src is None or src.empty:
                continue
            for period in ("day", "night"):
                row = src[(src["zone"] == zone) & (src["period"] == period)]
                pk = _pkg_trend(cid, zone, period)
                if row.empty or pk is None:
                    continue
                a = float(row.iloc[0]["sen_slope_per_decade"])
                b = float(pk.sen_slope_c_per_decade)
                worst = max(worst, abs(a - b)); n += 1
    check("Fig. 5 - NB03 panel statistics match composite_trend_statistics",
          worst <= tol, f"{n} slopes compared, worst difference {worst:.2e}")


def series_vs_package(cids, kind):
    """Worst difference between the NB03 cluster series drawn and the package city series.

    A cluster series is the mean of its cities' values (NB03 §7a); its SUHII is the mean urban
    core minus the mean BRA. The annual CR range (P5-P95) is checked too; the package holds
    no monthly percentiles, so the monthly band stays the NB03 one."""
    P0, keys = ((pkg("annual_series"), ["year"]) if kind == "annual"
                else (pkg("monthly_series"), ["year", "month"]))
    worst, n, gaps = 0.0, 0, 0
    for cid in cids:
        df, ddf, _, _ = tp.panel_inputs(tp.load_cluster(cid, read_source(STATS03)), kind)
        P = P0[P0.cluster25.astype(int) == cid]
        for per in ("day", "night"):
            m = lambda z, col="value_c": P[(P.zone == z) & (P.period == per)].groupby(keys)[col].mean()
            z = lambda zn: df[(df.zone == zn) & (df.period == per)].set_index(keys)
            pairs = [(z(zn).median_lst, m(zp)) for zn, zp in
                     (("urban_core", "uc"), ("bra", "bra"), ("city_region", "cr"))]
            pairs.append((ddf[ddf.period == per].set_index(keys).delta_lst, m("uc") - m("bra")))
            if kind == "annual":
                pairs += [(z("city_region").p05_lst, m("cr", "p05_lst")),
                          (z("city_region").p95_lst, m("cr", "p95_lst"))]
            for a, b in pairs:
                d = (a - b).abs()
                gaps += int(d.isna().sum()); worst = max(worst, float(d.max())); n += len(d)
    return worst, n, gaps


assert_trends_match_package(SELECTED_CLUSTERS)
_w5, _n5, _g5 = series_vs_package(SELECTED_CLUSTERS, "annual")
check("Fig. 5 - the annual series drawn equal the package (cluster means of annual_series)",
      _w5 <= 1e-9 and _g5 == 0,
      f"{_n5} values (UC, BRA, CR, CR P5/P95, SUHII), worst difference {_w5:.1e} °C, {_g5} gaps")

paths = render_cluster_grid(SELECTED_CLUSTERS, "annual", MAIN_FIG / "Fig_05_annual_trends",
                            trend_cell=_trend_cell_pkg)

#### Fig. 6 · Monthly trends, selected clusters

**Manuscript** 3.1 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_8_monthly_trend` · **Code** the cell below, restored from the v3.0 cell (`Notebook/publication_outputs.ipynb` @ `d2fef8e`, cell 34); the renderer is defined in the Fig. 5 cell · **Output** `main_figures/Fig_06_monthly_trends`

**Data** NB03 monthly cluster series and monthly statistics (permitted; the package holds annual composites only), the series asserted against package `monthly_series`

**Changes from v3.0** (plan 1a): output name only.

In [ ]:
begin("Fig. 6")
# ── Fig. 6 · Monthly trends (v3.0 Figure 8; cell restored from publication_outputs.ipynb
#    @ d2fef8e, cell 34) ──
# The v3.0 call, unchanged: the Fig. 5 renderer with the monthly half of the NB03 panel. The
# trend table comes from NB03's monthly statistics, as in v3.0, because the package holds
# annual composites only. The monthly series drawn are asserted against the package.
_w6, _n6, _g6 = series_vs_package(SELECTED_CLUSTERS, "monthly")
check("Fig. 6 - the monthly series drawn equal the package (cluster means of monthly_series)",
      _w6 <= 1e-9 and _g6 == 0,
      f"{_n6} values (UC, BRA, CR, SUHII), worst difference {_w6:.1e} °C, {_g6} gaps")

paths = render_cluster_grid(SELECTED_CLUSTERS, "monthly", MAIN_FIG / "Fig_06_monthly_trends")

#### Fig. 7 · Persistent heat events by cluster

**Manuscript** 3.1 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_11_heat_event_bars` · **Code** the cell below, restored from the v3.0 cell (`Notebook/publication_outputs.ipynb` @ `d2fef8e`, cell 46), itself NB04 §7c `heat_event_bars_combined_composite` · **Output** `main_figures/Fig_07_heat_event_bars`

**Data** package `city_year_summary` (ERA5-Land event counts and longest events; day and night event tiers)

**Changes from v3.0** (plan 1a): values from the package; bar borders from the day and night event tiers (v1.0.1 has no single tier); panel letters (a), (b); codes C1 … C25. The C1 merge with Fig. 8 is undone.

In [ ]:
begin("Fig. 7")
# ── Fig. 7 · Persistent heat events by cluster (v3.0 Figure 11; cell restored from
#    publication_outputs.ipynb @ d2fef8e, cell 46: the NB04 §7c heat_event_bars_combined_composite) ──
# The v3.0 definition, unchanged: two side-by-side bar groups, bars = cluster mean across its
# cities, error bars = city minimum to maximum, bar border = event tier, clusters ordered by
# their larger bar, one legend below, saved as v3.0 saved it. Changed only as the plan allows:
#   - values from the package (city_year_summary), not the NB04 cluster summary;
#   - the border tier: v1.0.1 has a day and a night event tier per city, not v3.0's single
#     status, so the day bar shows the cluster's most common day tier and the night bar its
#     night tier (ties go to the first alphabetically, NB04's rule);
#   - panel letters (a), (b) with the v3.0 panel titles; cluster codes C1 ... C25.
import matplotlib.patches as _mpatch
_cy = pkg("city_year_summary")
_SPEC = [("day_n_era5_persist_evts", "era5_day_n_persistent_events"),
         ("night_n_era5_persist_evts", "era5_night_n_persistent_events"),
         ("day_era5_longest_days", "era5_day_longest_event_days"),
         ("night_era5_longest_days", "era5_night_longest_event_days")]
_g7 = _cy.groupby("cluster25")
_clh = pd.DataFrame({"n_cities": _g7.size()})
for _stem, _col in _SPEC:
    _clh[f"{_stem}_mean"], _clh[f"{_stem}_min"], _clh[f"{_stem}_max"] = (
        _g7[_col].mean(), _g7[_col].min(), _g7[_col].max())
for _per in ("day", "night"):
    _clh[f"{_per}_tier"] = _g7[f"{_per}_selected_event_tier"].agg(lambda s: s.mode().iloc[0])
_clh = _clh.reset_index()
check("Fig. 7 - 56 cities in 16 clusters, the four ERA5 event fields complete",
      len(_cy) == 56 and len(_clh) == 16 and _cy[[c for _, c in _SPEC]].notna().all().all(),
      f"events {int(_cy.era5_day_n_persistent_events.min())}-{int(_cy.era5_day_n_persistent_events.max())} (day), "
      f"{int(_cy.era5_night_n_persistent_events.min())}-{int(_cy.era5_night_n_persistent_events.max())} (night)")
check("Fig. 7 - every cluster has a day and a night event tier for the bar borders",
      _clh[["day_tier", "night_tier"]].isin(["EXCELLENT", "GOOD", "FAIR"]).all().all(),
      "day " + ", ".join(f"C{r.cluster25} {r.day_tier[0]}" for r in _clh.itertuples())
      + "; night " + ", ".join(f"C{r.cluster25} {r.night_tier[0]}" for r in _clh.itertuples()))

_TIER_EC = {"EXCELLENT": "#1a9850", "GOOD": "#f77f00", "FAIR": "#999999"}
_DAY_C, _NGT_C = "#d73027", "#4575b4"

def _bar_group(ax, df_cl, day_col, ngt_col, ylabel, title, bw=0.35):
    _sv = (df_cl.dropna(subset=["cluster25"]).set_index("cluster25")
           [[f"{day_col}_mean", f"{ngt_col}_mean"]].fillna(0)
           .max(axis=1).sort_values(ascending=False))
    cids = _sv.index.tolist(); xs = np.arange(len(cids))
    for xi, cid in enumerate(cids):
        row = df_cl[df_cl["cluster25"] == cid].iloc[0]
        for col, xoff, cc, per in [(day_col, -bw / 2, _DAY_C, "day"), (ngt_col, +bw / 2, _NGT_C, "night")]:
            ec = _TIER_EC.get(str(row.get(f"{per}_tier", "")), "#cccccc")
            m  = float(row.get(f"{col}_mean", 0) or 0)
            lo = float(row.get(f"{col}_min", m) or m)
            hi = float(row.get(f"{col}_max", m) or m)
            ax.bar(xi + xoff, m, width=bw, color=cc, alpha=0.75,
                   edgecolor=ec, linewidth=1.4, zorder=3)
            ax.errorbar(xi + xoff, m, yerr=[[max(m - lo, 0)], [max(hi - m, 0)]],
                        fmt="none", ecolor="#333333", elinewidth=1.0, capsize=2.5, zorder=4)
    ax.set_xticks(xs)
    ax.set_xticklabels([f"C{int(c)}" for c in cids], fontsize=7, rotation=45, ha="right")
    ax.set_xlabel("Cluster", fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(title, fontsize=9, pad=4)
    ax.grid(axis="y", linestyle=":", alpha=0.45)
    ax.spines[["top", "right"]].set_visible(False)

fig11, (a11, b11) = plt.subplots(1, 2, figsize=(11.69, 5.85), gridspec_kw={"wspace": 0.28})
_bar_group(a11, _clh, "day_n_era5_persist_evts", "night_n_era5_persist_evts", "Events", "(a)  ERA5 Persistent Events")
_bar_group(b11, _clh, "day_era5_longest_days", "night_era5_longest_days", "Days", "(b)  ERA5 Longest Event")
_leg11 = [
    _mpatch.Patch(color=_DAY_C, alpha=0.75, label="Daytime"),
    _mpatch.Patch(color=_NGT_C, alpha=0.75, label="Night-time"),
    _mpatch.Patch(fc="w", ec="#1a9850", lw=1.5, label="EXCELLENT tier"),
    _mpatch.Patch(fc="w", ec="#f77f00", lw=1.5, label="GOOD tier"),
    _mpatch.Patch(fc="w", ec="#999999", lw=1.5, label="FAIR tier"),
]
# Drop the legend ~1 cm below the plots (clear of the rotated tick labels) with a
# "Legend" header, and keep the descriptive caption as a thin line beneath the box.
fig11.tight_layout(rect=(0, 0.06, 1, 1))
fig11.legend(handles=_leg11, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.017),
             fontsize=8, framealpha=0.9, columnspacing=1.4, handlelength=1.5,
             title="Legend", title_fontsize=9, alignment="center")
fig11.text(0.5, -0.135,
           "Bars = cluster mean across cities  ·  Error bars = city-level min–max  ·  "
           "Bar border = event tier of that period (most common in the cluster)",
           ha="center", va="top", fontsize=7, color="#555", style="italic")
_p11 = savefig(fig11, MAIN_FIG / "Fig_07_heat_event_bars", as_v30=True, show=True)

#### Fig. 8 · Event-scale SUHII by city

**Manuscript** 3.1 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_13_seviri_suhii_cities` · **Code** the cell below, restored from the v3.0 cell (`Notebook/publication_outputs.ipynb` @ `d2fef8e`, cell 47), itself NB04 §10d `create_suhii_city_chart` · **Output** `main_figures/Fig_08_seviri_suhii_cities`

**Data** package `city_year_summary` (`day_seviri_suhii_c`, `night_seviri_suhii_c`, `cluster25`)

**Changes from v3.0** (plan 1a): values from the package; no title; legend codes C1 … C25.

In [ ]:
begin("Fig. 8")
# ── Fig. 8 · Event-scale SUHII by city (v3.0 Figure 13; cell restored from
#    publication_outputs.ipynb @ d2fef8e, cell 47: NB04 §10d create_suhii_city_chart) ──
# The v3.0 call, unchanged: A4 portrait, day left and night right, cities sorted by daytime
# SUHII, bars filled by cluster colour, the cluster legend in 4 columns and the national box,
# saved as v3.0 saved it. Changed only as the plan allows:
#   - values from the package (city_year_summary SEVIRI fields), not the NB04 tables, in the
#     same city / period / suhii / note layout the function reads;
#   - no in-figure title; cluster codes "C1 ... C25" in the legend (v3.0 "Cl 1").
_cy8 = pkg("city_year_summary")
_sev8 = pd.concat([pd.DataFrame({"city": _cy8.city, "period": _per,
                                 "suhii": _cy8[f"{_per}_seviri_suhii_c"],
                                 "note": np.where(_cy8[f"{_per}_seviri_suhii_c"].notna(), "ok", "missing")})
                   for _per in ("day", "night")], ignore_index=True)
_clu8 = dict(zip(_cy8.city, _cy8.cluster25.astype(int)))
check("Fig. 8 - SEVIRI SUHII for all 56 study cities, day and night, in 16 clusters",
      len(_cy8) == 56 and (_sev8.note == "ok").sum() == 112 and len(set(_clu8.values())) == 16,
      "national day mean {:.2f} °C [{:.2f} to {:.2f}], night mean {:.2f} °C [{:.2f} to {:.2f}]".format(
          _cy8.day_seviri_suhii_c.mean(), _cy8.day_seviri_suhii_c.min(), _cy8.day_seviri_suhii_c.max(),
          _cy8.night_seviri_suhii_c.mean(), _cy8.night_seviri_suhii_c.min(), _cy8.night_seviri_suhii_c.max()))

fig13 = viz.create_suhii_city_chart(
    df_seviri      = _sev8,
    cluster_lookup = _clu8,
    output_dir     = None,
    dpi            = config.FIGURE_DPI,
    cl_legend_ncol = 4,
    title          = None,
    cl_label       = "C{}",
)
paths = savefig(fig13, MAIN_FIG / "Fig_08_seviri_suhii_cities", as_v30=True, show=True)

### 1.3 · Results 3.2 — surface hot and cold spots (Objective 2)

#### Table 2 · Significant spots by cluster

**Manuscript** 3.2 · **v3.0 reference** `_superseded_v3.0/tables/Table_5_cluster_spot_occurrence` · **Code** the cell below, restored from the v3.0 cell (`Notebook/publication_outputs.ipynb` @ `d2fef8e`, cell 52): `build_table5` and `_grouped_xlsx` in `src/pub_hotspot_tables.py` · **Output** `tables/Table_02_significant_spots_by_cluster` (.csv, .xlsx)

**Data** package `spots` (cluster25, city, period, spot_type, area_l50_km2, bh_significant)

**Changes from v3.0** (plan 1a): values from the package; codes C1 … C25; "hot spots" / "cold spots" in the group headings; title number; BH-FDR footnote. The other tables v3.0 cell 52 built are now built in the Table 4 cell.

In [ ]:
begin("Table 2")
# ── Table 2 · Significant spots by cluster (v3.0 Table 5; restored from publication_outputs.ipynb
#    @ d2fef8e, cell 52: src/pub_hotspot_tables.py, build_table5 and _grouped_xlsx) ──
# The v3.0 builder and writer, unchanged: one row per cluster and a national Total row; four
# groups (day / night x hot / cold) of spots, cities and L50 area; a flat CSV and a workbook with
# the banded two-row header, its title and its footnote. Changed only as the plan allows:
#   - values from the package (spots), not the NB05 summaries (build_table5(spots=...));
#   - cluster codes C1 ... C25 (v3.0 "Cl-1"); group headings "hot spots" / "cold spots";
#   - the workbook title carries the new number; the footnote adds the BH-FDR statement.
import pub_hotspot_tables as _pht
_sp2 = pkg("spots")
_HEAD2 = {"Sig. Hotspots": "Sig. hot spots", "Sig. Cool spots": "Sig. cold spots"}
def _head2(c):
    for a, b in _HEAD2.items():
        c = c.replace(a, b)
    return c

t5 = _pht.build_table5(None, spots=_sp2, cluster_fmt="C{}").rename(columns=_head2)
t5g = t5.copy(); t5g["_group"] = t5g["Cluster"].eq("Total")
_grp2 = [_head2(g) for _, _, g in _pht._T5_GROUPS]
_cl2, _tot2 = t5[t5.Cluster != "Total"], t5[t5.Cluster == "Total"].iloc[0]
check("Table 2 - cluster rows add up to the national row: spots and cities in all four groups",
      all(_cl2[f"{g}|{m}"].sum() == _tot2[f"{g}|{m}"] for g in _grp2
          for m in ("No. of spots", "No. of cities"))
      and sum(_tot2[f"{g}|No. of spots"] for g in _grp2) == len(_sp2),
      "; ".join(f"{g}: {int(_tot2[g + '|No. of spots'])} spots, {int(_tot2[g + '|No. of cities'])} cities"
                for g in _grp2) + f"; {len(_sp2)} spots in all")
_bhg = _sp2.groupby(["city", "period", "spot_type"]).bh_significant
check("Table 2 - BH-FDR survival is one value per city-period-type group",
      (_bhg.nunique() == 1).all(),
      f"{int(_sp2.bh_significant.sum())} of {len(_sp2)} spots, {int(_bhg.first().sum())} of {_bhg.ngroups} groups")
FOOT_T2 = ("Area = total L50 footprint (sq.km). Total row = national aggregate. "
           f"BH-FDR: {int(_sp2.bh_significant.sum())} of the {len(_sp2)} spots, in "
           f"{int(_bhg.first().sum())} of {_bhg.ngroups} city-period-type groups, survive "
           "Benjamini-Hochberg correction; survival is a property of the group, repeated on "
           "each of its spots, not a per-spot test.")

csv2 = TABLES / "Table_02_significant_spots_by_cluster.csv"
t5.columns = [c.split('|')[-1] if '|' in c else c for c in t5.columns]  # flat for csv
t5.to_csv(csv2, index=False)
_pht._grouped_xlsx(t5g, TABLES / "Table_02_significant_spots_by_cluster.xlsx",
                   id_cols=['Cluster'], groups=_grp2, subs=_pht._T5_METRICS,
                   title='Table 2: Summary of significant spots occurrence across PLT clusters (56 cities)',
                   footnote=FOOT_T2)

print(f"Saved: {csv2.name}, {csv2.with_suffix('.xlsx').name} in {TABLES.relative_to(ROOT)}")
_show2 = t5g.drop(columns="_group").set_index("Cluster")
_show2.columns = pd.MultiIndex.from_tuples([tuple(c.split("|")) for c in _show2.columns])
display(_show2)
print(FOOT_T2)

#### Fig. 9 · Surface hot and cold spots, all cities, day

**Manuscript** 3.2 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_15_spatial_extent_day` · **Code** the cell below, restored from the v3.0 cell (`Notebook/publication_outputs.ipynb` @ `d2fef8e`, cell 54); one loop writes Figs 9 and 10 · **Output** `main_figures/Fig_09_spot_maps_day`

**Data** NB05 A4 sheets and map legend (permitted); spot and contour counts asserted against package `spots` and `spot_contours`

**Changes from v3.0** (plan 1a): water lines lightened and thinned; output name.

In [ ]:
begin("Fig. 9")
# ── Figs 9 and 10 · Surface hot and cold spots, all cities, day and night (v3.0 Figures 15 + 16;
#    cell restored from publication_outputs.ipynb @ d2fef8e, cell 54) ──
# The v3.0 definition, unchanged: NB05's A4 sheets of all 56 cities (§11a2 plot_a4_grid) with
# their titles dropped, the north arrow moved and NB05's map legend added as a bottom strip,
# one loop for both periods. Changed only as the plan allows: the water lines are lightened and
# thinned; the output names are the new ones; the spot and contour counts NB05 drew are
# asserted against the package; the notebook shows the saved files.
import re as _re54mod
from lxml import etree as _ET
import fitz as _fitz
_SVGNS = "http://www.w3.org/2000/svg"
_re54 = _re54mod
_re54_I = _re54mod.IGNORECASE
_re54_W = _re54mod.compile(r'stroke-width:\s*([0-9.]+)')
_ECOMM = _ET.Comment("").__class__

def _gcomment(g):
    for ch in g:
        if isinstance(ch, _ECOMM):
            return (ch.text or "").strip()
    return ""

def _fig1516(src, dst_svg):
    tree = _ET.parse(str(read_source(src))); root = tree.getroot()
    _vb = (root.get("viewBox") or "0 0 575.30448 864.869096").split()
    W, H = float(_vb[2]), float(_vb[3])
    _title = ("Urban Heat Island", "KDE 50% contours")
    drop, arrow = [], None
    for g in root.iter(f"{{{_SVGNS}}}g"):
        gid = g.get("id", ""); c = _gcomment(g)
        if gid.startswith("text_") and any(m in c for m in _title):
            drop.append(g)
        if gid.startswith("axes_"):
            _txts = [ch for ch in g.iter(f"{{{_SVGNS}}}g") if ch.get("id", "").startswith("text_")]
            if _txts and all(_gcomment(t) == "N" for t in _txts):
                arrow = g
    for g in drop:
        g.getparent().remove(g)
    # Water and coastline: NB05 draws these in saturated cyan, which in the creek-laced
    # cities (Port Harcourt, Lagos, Warri) competes with the spots for attention. They
    # are lightened and thinned here, in the same post-processing pass that already
    # edits this SVG, so NB05 itself is untouched.
    # #00e5ff and #0095aa are the COLD-SPOT contour colours, not water: lightening
    # those would wash out the very features the figure is about. Water is the pale
    # blue fill #c6e8ff with the #7ec8e3 stroke, and the fine grey lines are roads.
    _WATER_IN = {"#7ec8e3"}
    _WATER_OUT = "#dcecf3"
    for _el in root.iter():
        _st = _el.get("style")
        if _st:
            _low = _st.lower()
            if any(_w in _low for _w in _WATER_IN):
                for _w in _WATER_IN:
                    _st = _re54.sub(_re54.escape(_w), _WATER_OUT, _st, flags=_re54_I)
                _st = _re54_W.sub(lambda m: "stroke-width:%.2f"
                                  % (float(m.group(1)) * 0.55), _st)
                _el.set("style", _st)
        _sc = _el.get("stroke")
        if _sc and _sc.lower() in _WATER_IN:
            _el.set("stroke", _WATER_OUT)
            if _el.get("stroke-width"):
                _el.set("stroke-width", "%.2f" % (float(_el.get("stroke-width")) * 0.55))
    if arrow is not None:                       # move N-arrow down to the colour-bar level
        dy = ((1 - 0.021) - (1 - 0.972)) * H
        arrow.set("transform", f"translate(-18.000,{dy:.3f}) {arrow.get('transform', '')}".strip())
    # composite the shared map legend into a new bottom strip
    _legW, _legH = 582.559375, 64.8
    root.set("height", f"{H + _legH}pt"); root.set("viewBox", f"0 0 {W} {H + _legH}")
    _defs = root.find(f"{{{_SVGNS}}}defs")
    _lr = _ET.parse(str(read_source(_F05 / "a4_grid_legend.svg"))).getroot()
    _ld = _lr.find(f"{{{_SVGNS}}}defs")
    _have = {c.get("id") for c in _defs if c.get("id")}
    if _ld is not None:
        for c in list(_ld):
            if not (c.get("id") and c.get("id") in _have):
                _defs.append(c)
    _lf = _lr.find(f"{{{_SVGNS}}}g[@id='figure_1']"); _lf.set("id", "legend_fig")
    _wrap = _ET.SubElement(root, f"{{{_SVGNS}}}g")
    _wrap.set("transform", f"translate({(W - _legW) / 2.0:.3f},{H:.3f})"); _wrap.append(_lf)
    dst_svg.parent.mkdir(parents=True, exist_ok=True)
    tree.write(str(dst_svg), xml_declaration=True, encoding="utf-8")
    dst_svg.with_suffix(".pdf").write_bytes(_fitz.open(str(dst_svg)).convert_to_pdf())

# the spots and contours in NB05's tables behind these sheets are the ones the package publishes
# (the sheets of 29 Oct draw only L50: NB05's cache path mislabels the L75 / L90 WKT's CRS)
_sp9, _sc9 = pkg("spots"), pkg("spot_contours")
_nb05sp = pd.read_csv(read_source(RESULTS_05 / "all_cities_spots.csv"))
_nb05ct = pyogrio.read_dataframe(read_source(RESULTS_05 / "master_thermal_spots.gpkg"),
                                 layer="hotspot_contours", read_geometry=False)

for _per, _no, _stem in [("day", "Fig. 9", "Fig_09_spot_maps_day"),
                         ("night", "Fig. 10", "Fig_10_spot_maps_night")]:
    begin(_no)
    _k = ["city", "spot_type"]
    _a, _b = (d[d.period == _per].groupby(_k).size().sort_index() for d in (_nb05sp, _sp9))
    check(f"{_no} - spots per city and type on the NB05 sheet equal the package ({_per})",
          _a.equals(_b), f"{int(_b.sum())} spots in {len(_b)} city-type groups")
    _a, _b = (d[d.period == _per].groupby(_k + ["contour_level"]).size().sort_index()
              for d in (_nb05ct, _sc9))
    check(f"{_no} - NB05 contour table (L50 / L75 / L90) per city and type equals the package ({_per})",
          _a.equals(_b), f"{int(_b.sum())} contours: " + ", ".join(
              f"{lv} {int(n)}" for lv, n in _b.groupby(level="contour_level").sum().items()))
    _src = _F05 / f"all_cities_hotspot_A4_{_per}.svg"
    _dst = MAIN_FIG / f"{_stem}.svg"
    if _src.exists():
        _fig1516(_src, _dst)
        show_file([_dst.with_suffix(".pdf"), _dst])
    else:
        print(f"{_no}: MISSING {_src.name}")

#### Fig. 10 · Surface hot and cold spots, all cities, night

**Manuscript** 3.2 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_16_spatial_extent_night` · **Code** the Fig. 9 cell (its second pass) · **Output** `main_figures/Fig_10_spot_maps_night`

**Data** as Fig. 9

**Changes from v3.0** (plan 1a): as Fig. 9.

#### Fig. 11 · Spot area by cluster (nested L50 / L75 / L90)

**Manuscript** 3.2 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_17_zone_area_distribution` · **Code** the cell below, restored from the v3.0 cell (`Notebook/publication_outputs.ipynb` @ `d2fef8e`, cell 56), itself NB05 §11c's nested bars · **Output** `main_figures/Fig_11_spot_area_by_cluster`

**Data** package `spots` (areas per contour level), asserted against `spot_contours`

**Changes from v3.0** (plan 1a): areas from the package (NB05's summary left L75 / L90 blank for 21 / 26 city groups); panel letters; "Hot spot"; legend title "Cluster".

In [ ]:
begin("Fig. 11")
# ── Fig. 11 · Spot area by cluster (v3.0 Figure 17; cell restored from publication_outputs.ipynb
#    @ d2fef8e, cell 56: NB05 §11c's nested L90 / L75 / L50 bars on a 2 x 2 page) ──
# The v3.0 definition, unchanged: one bar per cluster, sorted by L50 area, the three nested
# contour levels drawn as L90 / L75-L90 / L50-L75 segments in the cluster colour, two legends
# below, saved as v3.0 saved it. Changed only as the plan allows:
#   - areas from the package (spots), not NB05's spot_comprehensive_summary.csv, which leaves
#     L75 / L90 blank for 21 / 26 city groups that the package fills (the L50 areas agree);
#   - panel letters (a)-(d) before the v3.0 panel titles; "Hot spot"; legend title "Cluster".
from matplotlib.patches import Patch as _Patch17
_CLR = config.CLUSTER25_COLORS
_sp17, _sc17 = pkg("spots"), pkg("spot_contours")
_g17 = (_sp17.groupby(["spot_type", "period", "cluster25"])
        .agg(L50=("area_l50_km2", "sum"), L75=("area_l75_km2", "sum"),
             L90=("area_l90_km2", "sum"), n=("spot_uid", "size"))
        .reset_index()
        .rename(columns={"spot_type": "Type", "period": "Period", "cluster25": "Cluster"}))
check("Fig. 11 - every spot is in one bar, and L90 <= L75 <= L50 in every bar",
      int(_g17.n.sum()) == len(_sp17)
      and bool(((_g17.L90 <= _g17.L75 + 1e-9) & (_g17.L75 <= _g17.L50 + 1e-9)).all()),
      f"{int(_g17.n.sum())} spots in {len(_g17)} bars")
_lv17 = _sc17.groupby("contour_level").area_km2.sum()
check("Fig. 11 - the L50 / L75 / L90 totals equal spot_contours",
      all(abs(_g17[lv].sum() - _lv17[lv]) < 1e-6 for lv in ("L50", "L75", "L90")),
      ", ".join(f"{lv} {_g17[lv].sum():,.1f} km²" for lv in ("L50", "L75", "L90")))

_PANELS17 = [("hot", "day", "(a)  Hot spot — Day"), ("hot", "night", "(b)  Hot spot — Night"),
             ("cold", "day", "(c)  Cold spot — Day"), ("cold", "night", "(d)  Cold spot — Night")]
fig17, _axes17 = plt.subplots(2, 2, figsize=(11.69, 8.0))
_allcl17 = set()
for _ax, (_typ, _per, _title) in zip(_axes17.flat, _PANELS17):
    _sub = _g17[(_g17.Type == _typ) & (_g17.Period == _per)].copy()
    _sub = _sub[_sub["L50"] > 0].sort_values("L50", ascending=False)
    if _sub.empty:
        _ax.set_visible(False); continue
    _x = np.arange(len(_sub))
    for _xi, _r in zip(_x, _sub.itertuples()):
        _cl = int(_r.Cluster); _col = _CLR.get(_cl, "#888888"); _allcl17.add(_cl)
        _l90, _l75, _l50 = float(_r.L90), float(_r.L75), float(_r.L50)
        _ax.bar(_xi, _l90, color=_col, alpha=1.0, hatch="xxx", edgecolor="white", linewidth=0.3, zorder=3)
        _ax.bar(_xi, max(_l75 - _l90, 0), bottom=_l90, color=_col, alpha=0.72, hatch="//", edgecolor="white", linewidth=0.3, zorder=3)
        _ax.bar(_xi, max(_l50 - _l75, 0), bottom=_l75, color=_col, alpha=0.42, edgecolor="white", linewidth=0.3, zorder=3)
    _ax.set_xticks(_x); _ax.set_xticklabels([f"C{int(c)}" for c in _sub["Cluster"]], rotation=45, ha="right", fontsize=7.5)
    _ax.set_ylabel("Total area (km²)", fontsize=9)
    _ax.set_title(_title, fontsize=11, fontweight="bold", pad=5)
    _ax.grid(axis="y", ls=":", lw=0.4, alpha=0.5); _ax.spines[["top", "right"]].set_visible(False)
fig17.tight_layout(rect=(0, 0.11, 1, 1))
_zh17 = [_Patch17(facecolor="#555", alpha=1.0, hatch="xxx", edgecolor="white", label="L90 dense core"),
         _Patch17(facecolor="#555", alpha=0.72, hatch="//", edgecolor="white", label="L75 inner zone"),
         _Patch17(facecolor="#555", alpha=0.42, edgecolor="white", label="L50 outer extent")]
_lg17a = fig17.legend(handles=_zh17, title="Contour zone", loc="lower left", bbox_to_anchor=(0.02, 0.01),
                      ncol=1, fontsize=8, title_fontsize=9, framealpha=0.9)
_lg17a.get_title().set_fontweight("bold")
_ch17 = [_Patch17(facecolor=_CLR.get(c, "#888"), label=f"C{c}") for c in sorted(_allcl17)]
# _lg17b = fig17.legend(handles=_ch17, title="Cluster", loc="lower center", bbox_to_anchor=(0.60, 0.005),
#                      ncol=8, fontsize=7.5, title_fontsize=9, framealpha=0.9, columnspacing=1.0, handlelength=1.2)
#_lg17b.get_title().set_fontweight("bold")
_p17 = savefig(fig17, MAIN_FIG / "Fig_11_spot_area_by_cluster", as_v30=True, show=True)

### 1.4 · Results 3.3 — population within spots (Objective 3)

#### Fig. 12 · Population within hot and cold spots

**Manuscript** 3.3 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_20_population_exposure_panel` · **Code** the cell below

**Data** package `city_exposure_by_contour` (asserted); now plotted from NB05 `halfdonut_city_summary`, to come from the package

In [ ]:
begin("Fig. 12")
# ── Fig. 8 · Population within spots (was v3.0 Figure 20, §3 Discussion) ────
# Moved from the Discussion to Results. The v3.0 definition is kept whole: fresh maps
# with day/night half-donuts, synced colours, and the per-cluster half-donut panel.
# Changed for the SCS rebuild: the exposure numbers come from the package
# (city_exposure_by_contour), asserted against the NB05 half-donut summary, the output
# follows the new numbering, and there is no in-figure title.
# ── Figure 20 — FULLY REPLOTTED panel (maps drawn fresh, synced to half-donut colours) ──
import matplotlib.colors as _mc20
import matplotlib.patheffects as _pe20
from matplotlib.patches import Wedge as _W20, Patch as _P20
from matplotlib.lines import Line2D as _L20
_HOT20, _COLD20 = "#E07B00", "#2B6CB0"
_DAYF, _NIGHTF = 0.18, 0.48          # day/night shades shared by maps + half-donut Ring2
_halo20 = [_pe20.withStroke(linewidth=1.6, foreground="white")]
def _fmt20(v): return f"{v/1000:.1f}k" if v >= 1000 else f"{v:.0f}"
def _blend20(h, f):
    r, g, b = _mc20.to_rgb(h); return (r + (1 - r) * f, g + (1 - g) * f, b + (1 - b) * f)

_nigeria20 = gpd.read_file(read_source(ROOT / "data" / "inputs" / "geospatial_features" / "boundary_Nigeria.geojson"))
_hd20 = pd.read_csv(read_source(RESULTS_05 / "halfdonut_city_summary.csv"))

# ── the package is the source of record for every exposure number here ──
_ex20 = pkg("city_exposure_by_contour")
_w20 = 0.0
for _r in _ex20.itertuples():
    _m = _hd20[(_hd20.city == _r.city) & (_hd20.spot_type == _r.spot_type)
               & (_hd20.period == _r.period)]
    if _m.empty:
        continue
    _m = _m.iloc[0]
    for _pk, _nb in (("pop_l50_persons", "pop_50"), ("area_l50_km2", "area_km2_50")):
        _w20 = max(_w20, abs(float(getattr(_r, _pk)) - float(_m[_nb])))
check("Fig. 8 - NB05 half-donut summary matches city_exposure_by_contour",
      _w20 < 1e-6, f"worst difference {_w20:.2e} over {len(_ex20)} city rows")
_hotnight20 = _ex20[(_ex20.spot_type == "hot")
                    & (_ex20.period == "night")].pop_l50_persons.sum()
check("Fig. 8 - residents in night-time hot spots (L50)",
      round(_hotnight20) == 9_184_401, f"{round(_hotnight20):,}")
_ce20 = pd.read_csv(read_source(RESULTS_04 / "city_heat_event_summary.csv"))
_NCLU = _ce20.groupby("cluster25")["city"].count().to_dict()
_hd20["cluster"] = _hd20["city"].map(dict(zip(_ce20["city"], _ce20["cluster25"])))
_coords20 = {}
for _d in sorted(p for p in RESULTS_AOI.iterdir() if p.is_dir()):
    _uc = _d / f"{_d.name}_urban_core_4326_v5.geojson"
    if _uc.exists():
        _c = gpd.read_file(read_source(_uc)).geometry.union_all().centroid
        _coords20[_d.name.replace("_", " ")] = (_c.x, _c.y)

def _draw_map20(ax, spot_type, base, tag):
    _nigeria20.plot(ax=ax, facecolor="#f2f1ee", edgecolor="#333", lw=0.9, zorder=1)
    _nigeria20.boundary.plot(ax=ax, edgecolor="#222", lw=1.1, zorder=6)
    _dc, _nc = _blend20(base, _DAYF), _blend20(base, _NIGHTF)
    _sub = _hd20[_hd20.spot_type == spot_type]; _rows = {}
    for _city, (_lon, _lat) in _coords20.items():
        _s = _sub[_sub.city == _city]
        _da = float(_s[_s.period == "day"]["area_km2_50"].sum()); _na = float(_s[_s.period == "night"]["area_km2_50"].sum())
        _dp = float(_s[_s.period == "day"]["pop_50"].sum()); _np = float(_s[_s.period == "night"]["pop_50"].sum())
        _rows[_city] = dict(lon=_lon, lat=_lat, da=_da, na=_na, dp=_dp, npp=_np, ta=_da + _na, tp=_dp + _np)
    _maxp = max((r["tp"] for r in _rows.values() if r["tp"] > 0), default=1); _SC = 0.62
    for _city, r in _rows.items():
        if r["ta"] <= 0 or r["tp"] <= 0:
            ax.plot(r["lon"], r["lat"], "o", color="#c9c9c9", ms=2.5, zorder=5, alpha=0.7); continue
        r["rad"] = _SC * (max(r["tp"], 1) / _maxp) ** 0.5; _df = r["da"] / r["ta"]
        if r["da"] > 0: ax.add_patch(_W20((r["lon"], r["lat"]), r["rad"], 90, 90 + _df * 360, facecolor=_dc, edgecolor="white", lw=0.6, zorder=6))
        if r["na"] > 0: ax.add_patch(_W20((r["lon"], r["lat"]), r["rad"], 90 + _df * 360, 450, facecolor=_nc, edgecolor="white", lw=0.6, zorder=6))
        ax.plot(r["lon"], r["lat"], "ko", ms=1.6, zorder=7)
    _placed = []
    for _city, r in sorted(_rows.items(), key=lambda kv: -kv[1]["tp"]):
        if r["tp"] < 500 or "rad" not in r or len(_placed) >= 13: continue
        _px, _py = r["lon"], r["lat"] - r["rad"] - 0.05
        if any((_px - qx) ** 2 + (_py - qy) ** 2 < 0.85 ** 2 for qx, qy in _placed): continue
        ax.text(_px, _py, f"{_city}\n☀{_fmt20(r['dp'])} ☾{_fmt20(r['npp'])}", ha="center", va="top",
                fontsize=6.5, color="#1a1a1a", zorder=8, linespacing=0.95,
                path_effects=_halo20)
        _placed.append((_px, _py))
    _b = _nigeria20.total_bounds
    ax.set_xlim(_b[0] - 0.4, _b[2] + 0.4); ax.set_ylim(_b[1] - 0.8, _b[3] + 0.6); ax.set_aspect("equal"); ax.axis("off")
    _ml = (_b[1] + _b[3]) / 2; _dx = 200 / (111.32 * np.cos(np.radians(_ml))); _x0 = _b[0] + 0.2; _y0 = _b[1] - 0.55
    ax.plot([_x0, _x0 + _dx], [_y0, _y0], color="#111", lw=2.2, zorder=9, solid_capstyle="butt")
    ax.text(_x0 + _dx / 2, _y0 + 0.12, "200 km", ha="center", va="bottom",
            fontsize=6.5, zorder=9)
    ax.annotate("", xy=(_b[2] + 0.05, _b[3] + 0.35), xytext=(_b[2] + 0.05, _b[3] - 0.25), arrowprops=dict(arrowstyle="->", lw=1.1, color="#111"), zorder=9)
    ax.text(_b[2] + 0.05, _b[3] + 0.42, "N", ha="center", va="bottom", fontsize=6.5, fontweight="bold", zorder=9)
    ax.set_title(tag, fontsize=8.5, fontweight="bold", pad=2, loc="left")

def _compact_halfdonut(ax, df, title=None, fs=7.0):
    r_i, r_w = [0.32, 0.60, 0.92], [0.25, 0.28, 0.28]
    def pop(st, per, col="pop_50"):
        s = df[(df.spot_type == st) & (df.period == per)]; return float(s[col].sum()) if len(s) else 0.0
    th = pop("hot", "day") + pop("hot", "night"); tc = pop("cold", "day") + pop("cold", "night"); tot = (th + tc) or 1
    hs, cs = 180 * th / tot, 180 * tc / tot
    def put(ang, r, t, size):
        a = np.radians(ang); ax.text(r * np.cos(a), r * np.sin(a), t, ha="center", va="center", fontsize=size, color="#1a1a1a", fontweight="bold", zorder=20, path_effects=_halo20)
    for st, ba, span, base in (("hot", 180 - hs, hs, _HOT20), ("cold", 0, cs, _COLD20)):
        stt = th if st == "hot" else tc
        if span < 0.5: continue
        ax.add_patch(_W20((0, 0), r_i[0] + r_w[0], ba, ba + span, width=r_w[0], facecolor=_blend20(base, 0.0), edgecolor="white", lw=1.2))
        # ring 1 carries no number either: it would sum day and night.
        ang2, s2 = ba, stt or 1
        for per, fade in (("day", _DAYF), ("night", _NIGHTF)):
            pp = pop(st, per); sw2 = span * (pp / s2)
            if pp <= 0 or sw2 <= 0.5:
                ang2 += sw2; continue
            ax.add_patch(_W20((0, 0), r_i[1] + r_w[1], ang2, ang2 + sw2, width=r_w[1], facecolor=_blend20(base, fade), edgecolor="white", lw=1.0))
            if sw2 >= 20: put(ang2 + sw2 / 2, r_i[1] + r_w[1] / 2, _fmt20(pp), fs + 0.3)
            p90 = min(pop(st, per, "pop_90"), pp); p75 = min(pop(st, per, "pop_75"), pp); ang3 = ang2
            for seg, f3 in ((p90, 0.62), (max(0, p75 - p90), 0.74), (max(0, pp - p75), 0.85)):
                sw3 = sw2 * (seg / pp) if pp > 0 else 0
                if seg > 0 and sw3 > 0.3:
                    ax.add_patch(_W20((0, 0), r_i[2] + r_w[2], ang3, ang3 + sw3, width=r_w[2], facecolor=_blend20(base, f3), edgecolor="white", lw=0.6))
                    if sw3 >= 30: put(ang3 + sw3 / 2, r_i[2] + r_w[2] / 2, _fmt20(seg), fs - 1.3)
                    ang3 += sw3
            ang2 += sw2
    # No centre total: it would add hot to cold and day to night, counting the same
    # residents several times. Every number shown belongs to one spot type and one
    # period. tot survives only as the geometry that sets the two half spans.
    if title: ax.text(0, 1.58, title, ha="center", va="center", fontsize=max(6.5, fs + 2), fontweight="bold")
    ax.set_xlim(-1.32, 1.32); ax.set_ylim(-0.22, 1.82); ax.axis("off"); ax.set_aspect("equal")

def _agg20(sub): return sub.groupby(["spot_type", "period"])[["pop_50", "pop_75", "pop_90"]].sum().reset_index()
def _cltitle20(cid): return f"Cluster {cid} ({_NCLU.get(cid, 0)} cities)"

fig20 = plt.figure(figsize=(W_FULL, 9.3))
_draw_map20(fig20.add_axes([0.015, 0.70, 0.475, 0.29]), "hot", _HOT20, "(a) Population within hot spots")
_draw_map20(fig20.add_axes([0.51, 0.70, 0.475, 0.29]), "cold", _COLD20, "(b) Population within cold spots")
_POS20 = {"C25": [0.02, 0.50, 0.36, 0.155], "C4": [0.62, 0.50, 0.36, 0.155], "NAT": [0.32, 0.34, 0.36, 0.16],
          "C9": [0.02, 0.175, 0.36, 0.155], "C13": [0.62, 0.175, 0.36, 0.155]}
_CID20 = {"C25": 25, "C4": 4, "C13": 13, "C9": 9}
for _k, _rect in _POS20.items():
    _ax = fig20.add_axes(_rect)
    if _k == "NAT":
        _compact_halfdonut(_ax, _agg20(_hd20), title="(c) National", fs=7.6)
    else:
        _compact_halfdonut(_ax, _agg20(_hd20[_hd20.cluster == _CID20[_k]]), title=_cltitle20(_CID20[_k]), fs=6.5)
_axl = fig20.add_axes([0.0, 0.0, 1.0, 0.14]); _axl.axis("off")
_h1 = [_P20(fc=_blend20(_HOT20, _DAYF), label="Hot · Day"), _P20(fc=_blend20(_HOT20, _NIGHTF), label="Hot · Night"),
       _P20(fc=_blend20(_COLD20, _DAYF), label="Cold · Day"), _P20(fc=_blend20(_COLD20, _NIGHTF), label="Cold · Night")]
_lg1 = _axl.legend(handles=_h1, loc="center left", bbox_to_anchor=(0.02, 0.62), ncol=2, fontsize=7.5,
                   title="Spot type × period  (darker = Day · lighter = Night)", title_fontsize=8, framealpha=0.9)
_lg1.get_title().set_fontweight("bold"); _axl.add_artist(_lg1)
_h2 = [_P20(fc=_blend20("#888", 0.62), label="L90 dense core"), _P20(fc=_blend20("#888", 0.74), label="L75 inner zone"),
       _P20(fc=_blend20("#888", 0.85), label="L50 outer extent"),
       _L20([0], [0], marker="o", mfc="none", mec="#555", ls="none", ms=10, label="circle ∝ pop (maps)")]
_lg2 = _axl.legend(handles=_h2, loc="center right", bbox_to_anchor=(0.98, 0.62), ncol=2, fontsize=7.5,
                   title="Half-donut contour rings  ·  map symbols", title_fontsize=8, framealpha=0.9)
_lg2.get_title().set_fontweight("bold")
_axl.text(0.5, 0.06, "Maps: city wedge split = Day/Night spot area, circle size ∝ population exposed  ·  "
          "Half-donuts: rings = Type → Day/Night → Contour; ☀ = day  ☾ = night",
          ha="center", va="bottom", fontsize=6.8, color="#444", transform=_axl.transAxes)
_p20 = savefig(fig20, MAIN_FIG / "Fig_12_population_within_spots")
plt.show(); print("Fig. 8 ->", ", ".join(p.name for p in _p20))

### 1.5 · Results 3.4 — stakeholder accounts (Objective 4)

#### Table 3 · Stakeholder accounts against the satellite record

**Manuscript** 3.4 · **v3.0 reference** `_superseded_v3.0/tables/Table_4_stakeholder_corroboration` (city, cluster and national rows) · **Code** the cell below

**Data** package `stakeholder_perception`, `trend_statistics`; NB03 agreement table (not in the package)

In [ ]:
begin("Table 3")
# ── Table 3 · Stakeholder accounts against the satellite record (was v3.0 Table 4) ─
# v3.0 wrote out Stakeholder_Validation_Comprehensive.csv as it stood, with city,
# cluster and national rows together. Changed for the SCS rebuild:
#   - cluster rows only; the city rows move to Supp. Table S6;
#   - the trend columns are counts of MEMBER CITIES whose urban-core Mann-Kendall
#     trend is a significant increase, which is the same per-city basis the agreement
#     percentages already use. The cluster-composite slope is not shown, because it
#     cannot be compared with a percentage computed city by city;
#   - cluster names from the approved labels, and single-stakeholder cities flagged.
comp = pd.read_csv(read_source(RESULTS_03 / "Stakeholder_Validation_Comprehensive.csv"))
_pc3 = pkg("stakeholder_perception")
_ts3 = pkg("trend_statistics")

_per_city3 = _pc3.groupby("city").size()
_singles3 = sorted(_per_city3[_per_city3 == 1].index)
_all_cities3 = sorted(_pc3.city.unique())


def _sig_inc(cities, period):
    """Member cities whose urban-core trend is a significant increase (p < 0.05)."""
    g = _ts3[(_ts3.zone == "uc") & (_ts3.period == period) & (_ts3.city.isin(cities))]
    return int(((g.mk_trend == "increasing") & (g.mk_p_value < 0.05)).sum()), g.city.nunique()


_rows3 = []
for _c in sorted(_pc3.cluster25.unique()):
    _g = _pc3[_pc3.cluster25 == _c]
    _cities = sorted(_g.city.unique())
    _v = comp[(comp.Level == "Cluster")
              & (comp.Cluster25.astype(str).str.lstrip("C0").str.lstrip("C")
                 == str(int(_c)))]
    _vr = _v.iloc[0] if len(_v) else None
    _ds, _dn = _sig_inc(_cities, "day")
    _ns, _nn = _sig_inc(_cities, "night")
    _sc = sorted(set(_cities) & set(_singles3))
    _rows3.append({
        "Cluster": CL.loc[int(_c)].cluster25,
        "Typology": CLUSTER_LABEL_LONG[int(_c)],
        "Stakeholders (n)": len(_g), "Cities (n)": len(_cities),
        "Single-stakeholder cities": ", ".join(_sc) if _sc else "\u2014",
        "Cities with significant urban-core increase, day": f"{_ds}/{_dn}",
        "Cities with significant urban-core increase, night": f"{_ns}/{_nn}",
        "Perceived warming, agreement (%)": (round(float(_vr["Q61_Annual_Agreement_%"]), 1)
                                             if _vr is not None else np.nan),
        "Perceived peak period": _vr["Q58_Peak_Response"] if _vr is not None else "",
        "Peak-month agreement (%)": (round(float(_vr["Q58_Monthly_Agreement_%"]), 1)
                                     if _vr is not None else np.nan),
        "Consensus": _vr["Consensus_Level"] if _vr is not None else ""})
tab3 = pd.DataFrame(_rows3)

_nat3 = comp[comp.Level == "National"].iloc[0]
_nd, _ndn = _sig_inc(_all_cities3, "day")
_nn3, _nnn = _sig_inc(_all_cities3, "night")
_npeak = int(round(float(_nat3["Q58_Monthly_Agreement_%"]) / 100.0 * 105))
check("Table 3 - the stakeholder cohort", tab3["Stakeholders (n)"].sum() == 105
      and len(tab3) == 10, f"{tab3['Stakeholders (n)'].sum()} stakeholders, "
      f"{len(tab3)} clusters, {len(_singles3)} single-stakeholder cities")
check("Table 3 - significant urban-core increases, per city",
      (_nd, _ndn) == (12, 21) and (_nn3, _nnn) == (14, 21),
      f"day {_nd}/{_ndn}, night {_nn3}/{_nnn} cities")
check("Table 3 - peak month held by 70 of 105 participants", _npeak == 70, str(_npeak))

tab3 = pd.concat([tab3, pd.DataFrame([{
    "Cluster": "All", "Typology": "All typologies with stakeholders",
    "Stakeholders (n)": 105, "Cities (n)": 21,
    "Single-stakeholder cities": f"{len(_singles3)} cities",
    "Cities with significant urban-core increase, day": f"{_nd}/{_ndn}",
    "Cities with significant urban-core increase, night": f"{_nn3}/{_nnn}",
    "Perceived warming, agreement (%)": round(float(_nat3["Q61_Annual_Agreement_%"]), 1),
    "Perceived peak period": _nat3["Q58_Peak_Response"],
    # computed from the count, not read from the CSV, which stores a truncated 66.6
    "Peak-month agreement (%)": round(100.0 * _npeak / 105, 1),
    "Consensus": f"{_npeak} of 105 participants"}])], ignore_index=True)

FOOT_T3 = ("Trend columns count the member cities with stakeholders whose urban-core "
           "Mann-Kendall trend is a significant increase (p < 0.05), the same per-city "
           "basis as the agreement percentages. City rows, each with its own trend, are "
           "in Supplementary Table S6.")

csv3 = TABLES / "Table_03_stakeholder_vs_satellite_cluster.csv"
tab3.to_csv(csv3, index=False)
xls3 = TABLES / "Table_03_stakeholder_vs_satellite_cluster.xlsx"
wb = xlsxwriter.Workbook(str(xls3)); ws = wb.add_worksheet("Table 3")
hdr  = wb.add_format({"bold": True, "bg_color": "#404040", "font_color": "white",
                      "border": 1, "align": "center", "valign": "vcenter", "text_wrap": True})
cell = wb.add_format({"border": 1, "valign": "vcenter", "font_size": 9})
foot = wb.add_format({"italic": True, "font_size": 8})
for j, c in enumerate(tab3.columns):
    ws.write(0, j, c, hdr)
for i, (_, r) in enumerate(tab3.iterrows(), start=1):
    for j, c in enumerate(tab3.columns):
        ws.write(i, j, "" if pd.isna(r[c]) else r[c], cell)
ws.write(len(tab3) + 2, 0, FOOT_T3, foot)
ws.set_column(0, 0, 9); ws.set_column(1, 1, 52); ws.set_column(2, 3, 11)
ws.set_column(4, 4, 26); ws.set_column(5, 6, 18); ws.set_column(7, 10, 14); ws.set_row(0, 34)
wb.close()
print("Table 3:", len(tab3), "rows ->", csv3.name, "+", xls3.name)
print("Footnote:", FOOT_T3)
display(tab3)

#### Fig. 13 · Stakeholder perception by cluster

**Manuscript** 3.4 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_9_stakeholder_perception.svg` · **Code** the cell below

**Data** package `stakeholder_perception`

> Until the Fig. 13 and Fig. 14 steps the Fig. 14 cell composes both into the C1 merge (`Fig_13_stakeholder_accounts`).

In [ ]:
begin("Fig. 13")
# ── Fig. 9 (a)–(c) · Stakeholder perception (was v3.0 Figure 9, §2.2) ─────
# v3.0 collected a finished artefact from NB03. It is now drawn here by the SAME
# function NB03 calls, viz.create_cluster_perception_chart, with its styling untouched.
# Changed for the SCS rebuild:
#   - the responses come from stakeholder_perception (package), not from NB03's cache;
#   - rows are named with the approved cluster labels;
#   - no in-figure title.
# Supp. Fig. S12 (city-level perception) still collects NB03's city panel unchanged.
_pc9 = pkg("stakeholder_perception")
check("Fig. 9a-c - the perception cohort",
      len(_pc9) == 105 and _pc9.city.nunique() == 21 and _pc9.cluster25.nunique() == 10,
      f"{len(_pc9)} participants, {_pc9.city.nunique()} cities, "
      f"{_pc9.cluster25.nunique()} clusters")
check("Fig. 9a-c - no perception item carries a null",
      int(_pc9[["perceived_warming_10yr", "perceived_diurnal_severity",
                "perceived_peak_period"]].isna().sum().sum()) == 0,
      "all three items share one vocabulary in v1.0.1")

# the chart prefixes each row with "Cluster ", so the label carries only code + name
_row9 = {int(c): f"{CL.loc[int(c)].cluster25}  {CL.loc[int(c)].label_short}"
         for c in sorted(_pc9.cluster25.unique())}
_order9 = [_row9[c] for c in sorted(_pc9.cluster25.unique(),
                                    key=lambda c: -int((_pc9.cluster25 == c).sum()))]
_pc9 = _pc9.assign(_row=_pc9.cluster25.astype(int).map(_row9))

_w9 = pd.crosstab(_pc9._row, _pc9.perceived_warming_10yr).reindex(_order9)
_d9 = pd.crosstab(_pc9._row, _pc9.perceived_diurnal_severity).reindex(_order9)
_p9 = pd.crosstab(_pc9._row, _pc9.perceived_peak_period).reindex(_order9)

fig9ac, _fig9_donuts = viz.create_cluster_perception_chart(
    warming_by_cluster = _w9,
    diurnal_by_cluster = _d9,
    warming_total      = _pc9.perceived_warming_10yr.value_counts(),
    diurnal_total      = _pc9.perceived_diurnal_severity.value_counts(),
    peak_by_cluster    = _p9,
    peak_total         = _pc9.perceived_peak_period.value_counts(),
    cluster25_colors   = config.CLUSTER25_COLORS,
    # three rows of one panel, so the figure is built at the 190 mm print width
    figsize            = (W_FULL, 6.25),
    stacked            = True,
    legend_scale       = 1.0,
    suptitle           = False,
)
for _ax, _lt in zip(fig9ac.axes, ("(a)", "(b)", "(c)")):
    _ax.set_title(_lt, loc="left", fontsize=11, fontweight="bold")
_FIG9AC = MAIN_FIG / "_fig09_panels_abc"
_p9ac = savefig(fig9ac, _FIG9AC, fit=W_FULL)
plt.show()
print("Fig. 9 (a)-(c) ->", ", ".join(p.name for p in _p9ac))

#### Fig. 14 · Hit rate against buffer distance

**Manuscript** 3.4 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_18_stakeholder_hitrate_national.svg` · **Code** the cell below

**Data** package `hitrate_sensitivity` (cohort basis, 55 / 39, row-based)

In [ ]:
begin("Fig. 14")
# ── Fig. 9 (d) · Hit-rate sensitivity (was v3.0 Figure 18, collected from NB05) ─
# v3.0 copied NB05's finished stakeholder_hitrate_national artefact, which rests on the
# superseded attendance basis. NB05 is left untouched, so its _draw_sensitivity
# definition is reproduced here unchanged in style and fed from hitrate_sensitivity
# (package), which is on the 105-participant cohort and uses the released coordinates.
import matplotlib.ticker as mticker
_DAY_FILL, _NIGHT_FILL, _FILL_A = "#F2C14E", "#9BA3AE", 0.25
STAKE_BUFFER_M = getattr(config, "STAKEHOLDER_VALIDATION_BUFFER_M", 1000)

sens_df = pkg("hitrate_sensitivity")
_a500 = sens_df[sens_df.buffer_m == 500].groupby(["spot_type", "period"])[["n_hit", "n_total"]].sum()
_want = {("hot", "day"): (4, 55), ("hot", "night"): (3, 55),
         ("cold", "day"): (4, 39), ("cold", "night"): (0, 39)}
_got = {k: (int(_a500.loc[k, "n_hit"]), int(_a500.loc[k, "n_total"])) for k in _want}
check("Fig. 9d - hit rates at 500 m on the cohort points", _got == _want,
      "; ".join(f"{k[0]} {k[1]} {v[0]}/{v[1]}" for k, v in _got.items()))


def _draw_sensitivity(chart_type):
    """NB05 §9's definition, unchanged apart from reading the package."""
    fig_s, (ax_hot, ax_cold) = plt.subplots(1, 2, figsize=(W_FULL, 2.55),
                                            sharey=False)
    fig_s.subplots_adjust(bottom=0.30, wspace=0.30)
    _panels = [("hot", ax_hot), ("cold", ax_cold)]
    for _st, _ax in _panels:
        _sub = sens_df[sens_df.spot_type == _st]
        _clr_spot = "#CC2222" if _st == "hot" else "#1155AA"
        _ax.set_xlabel("Buffer distance (km)", fontsize=7.5)
        _ax.set_ylabel("Hit-rate (%)", fontsize=7.5)
        _ax.set_title(("(d)  Hot spots" if _st == "hot" else "Cold spots"),
                      fontsize=8.5, fontweight="bold", color=_clr_spot, loc="left")
        _ax.grid(True, alpha=0.2, ls="--")
        _ax.tick_params(labelsize=6.5)
        _ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
        _ax.set_ylim(-3, 108)
        _ax.axvline(STAKE_BUFFER_M / 1000, color="#777", lw=0.9, ls=":", zorder=1)
        _ax2 = _ax.twinx()
        _ax2.set_ylabel("Total hit count", fontsize=7.5, color="#333")
        _ax2.tick_params(axis="y", labelsize=6.5, labelcolor="#333")
        for _per, _fill_clr in (("day", _DAY_FILL), ("night", _NIGHT_FILL)):
            _psub = _sub[_sub.period == _per]
            if not len(_psub):
                continue
            _tot = (_psub.groupby("buffer_m")[["n_hit", "n_total"]].sum()
                    .reset_index().sort_values("buffer_m"))
            _bx = _tot["buffer_m"].values / 1000
            _hy = _tot["n_hit"].values
            _ax2.fill_between(_bx, 0, _hy, alpha=_FILL_A, color=_fill_clr, zorder=1)
            _ax2.plot(_bx, _hy, color=_fill_clr, lw=0.7, alpha=0.6, zorder=1)
        if not len(_sub):
            _ax.text(0.5, 0.5, "No data", ha="center", va="center",
                     transform=_ax.transAxes, color="#aaa", fontsize=11)
            continue
        _line_handles = []
        if chart_type == "cluster":
            _cl_agg = (_sub.dropna(subset=["cluster25"])
                       .groupby(["buffer_m", "cluster25", "period"])
                       [["n_hit", "n_total"]].sum().reset_index())
            _cl_agg["hit_rate_cl"] = _cl_agg["n_hit"] / _cl_agg["n_total"]
            _active = sorted(_cl_agg.groupby("cluster25")["n_hit"].sum()
                             .loc[lambda s: s > 0].index.tolist())
            for _cl in _active:
                _clr_cl = config.CLUSTER25_COLORS.get(int(_cl), "#888")
                for _per, _ls in (("day", "-"), ("night", "--")):
                    _cs = (_cl_agg[(_cl_agg.cluster25 == _cl)
                                   & (_cl_agg.period == _per)].sort_values("buffer_m"))
                    if not len(_cs):
                        continue
                    _ax.plot(_cs["buffer_m"] / 1000, _cs["hit_rate_cl"] * 100,
                             color=_clr_cl, lw=1.5, ls=_ls, zorder=3)
                _line_handles.append(plt.Line2D([0], [0], color=_clr_cl, lw=1.5,
                                                label=f"C{int(_cl)}"))
        else:
            for _per, _ls, _lw in (("day", "-", 2.5), ("night", "--", 2.0)):
                _ns = (_sub[_sub.period == _per].groupby("buffer_m")[["n_hit", "n_total"]]
                       .sum().reset_index().sort_values("buffer_m"))
                if not len(_ns):
                    continue
                _ns["hr"] = _ns["n_hit"] / _ns["n_total"]
                _ax.plot(_ns["buffer_m"] / 1000, _ns["hr"] * 100, color=_clr_spot,
                         lw=_lw, ls=_ls, zorder=4)
    _leg_extra = [
        plt.Line2D([0], [0], ls="-", color="#555", lw=1.3, label="Daytime"),
        plt.Line2D([0], [0], ls="--", color="#555", lw=1.3, label="Nighttime"),
        plt.Rectangle((0, 0), 1, 1, fc=_DAY_FILL, alpha=_FILL_A + 0.1,
                      label="Daytime total hits"),
        plt.Rectangle((0, 0), 1, 1, fc=_NIGHT_FILL, alpha=_FILL_A + 0.1,
                      label="Nighttime total hits"),
        plt.Line2D([0], [0], ls=":", color="#777", lw=1.0,
                   label=f"{STAKE_BUFFER_M/1000:g} km"),
    ]
    fig_s.legend(handles=_leg_extra, loc="lower center", ncol=5, fontsize=6.5,
                 framealpha=0.9, bbox_to_anchor=(0.5, -0.02))
    return fig_s


fig9d = _draw_sensitivity("national")
_FIG9D = MAIN_FIG / "_fig09_panel_d"
_p9d = savefig(fig9d, _FIG9D, fit=W_FULL)
plt.show()
print("Fig. 9 (d) ->", ", ".join(p.name for p in _p9d))

# ── compose Fig. 9 from the two definitions, each at its own size ──
import fitz as _fitz9
_top = _fitz9.open(str(_FIG9AC.with_suffix(".pdf")))
_bot = _fitz9.open(str(_FIG9D.with_suffix(".pdf")))
_tw, _th = _top[0].rect.width, _top[0].rect.height
_bw, _bh = _bot[0].rect.width, _bot[0].rect.height
# Both parts are saved at the print width, so the page is pinned to it and neither
# is scaled: scaling a page shrinks its type, which is what pushed this figure below
# 6.5 pt before.
_W9 = 190 / 25.4 * 72.0
_sb = _W9 / _bw
_st9 = _W9 / _tw
_H9 = _th * _st9 + _bh * _sb
_out9 = _fitz9.open()
_pg = _out9.new_page(width=_W9, height=_H9)
_pg.show_pdf_page(_fitz9.Rect((_W9 - _tw * _st9) / 2, 0,
                              (_W9 + _tw * _st9) / 2, _th * _st9), _top, 0)
_pg.show_pdf_page(_fitz9.Rect((_W9 - _bw * _sb) / 2, _th * _st9,
                              (_W9 + _bw * _sb) / 2, _H9), _bot, 0)
_fig9_pdf = MAIN_FIG / "Fig_13_stakeholder_accounts.pdf"
_out9.save(str(_fig9_pdf))
_fig9_svg = MAIN_FIG / "Fig_13_stakeholder_accounts.svg"
# get_svg_image() returns a bare <svg> root; the declaration is added so the file is
# the XML its extension claims, like every other figure this notebook writes.
_svg9 = _out9[0].get_svg_image()
if not _svg9.lstrip().startswith("<?xml"):
    _svg9 = '<?xml version="1.0" encoding="utf-8" standalone="no"?>' + chr(10) + _svg9
_fig9_svg.write_text(_svg9, encoding="utf-8")
_out9.close(); _top.close(); _bot.close()
for _tmp in (_FIG9AC, _FIG9D):
    for _ext in (".pdf", ".svg"):
        _tmp.with_suffix(_ext).unlink(missing_ok=True)
check("Fig. 9 - composed from both definitions at their own sizes",
      _fig9_pdf.exists() and _fig9_pdf.open("rb").read(5) == b"%PDF-",
      f"{_W9:.0f} x {_H9:.0f} pt")
print("Fig. 9 ->", _fig9_pdf.name, "+", _fig9_svg.name)

### 1.6 · Results 3.5 — typology and the thermal anomalies (Objective 5)

#### Table 4 · Typology–anomaly association

**Manuscript** 3.5 · **v3.0 reference** `_superseded_v3.0/tables/Table_6_plt_thermal_association` · **Code** the cell below (builds it with the other tables of v3.0 cell 52)

**Data** NB06 association caches (the declared exception)

> Until their own steps this cell's `build_hotspot_tables()` call (v3.0 cell 52, moved here from the Table 2 cell) also writes Table 5 and Supp. Tables S5, S7 and S8.

In [ ]:
begin("Table 4")
# ── Tables (§3.3 + §3.4): Supp 6, Supp 7, Table 6, Table 7, Supp 8 ──
# Built together from cached NB05/NB06 CSVs into manuscript layouts (plain .csv + styled .xlsx)
# by src/pub_hotspot_tables.py (v3.0 cell 52). Moved here from the Table 2 cell, which now
# builds Table 2 (v3.0 Table 5) from the package; until their own steps this writes Tables 4
# and 5 and Supp. Tables S5, S7 and S8.
import pub_hotspot_tables as _pht
_tbls = _pht.build_hotspot_tables(read_source(RESULTS_05), read_source(RESULTS_06T), TABLES,
                                  table5=False)
for _k, _v in _tbls.items():
    print(f"  {_k:14} {_v.shape[0]:>3} rows x {_v.shape[1]} cols")
# ── SCS numbering: v3.0 Tables 6+7 become Table 4 ──
import shutil as _sh52
for _old, _new in (("Table_6_plt_thermal_association", "Table_04_typology_association"),
                   ("Supp_Table_6_city_zone_footprints", "Supp_Table_S5_spots_per_city"),
                   ("Supp_Table_7_buffer_sensitivity", "Supp_Table_S7_hit_rate_by_city"),
                   ("Supp_Table_8_best_element_attribution_full",
                    "Supp_Table_S8_best_element_attribution")):
    for _ext in (".csv", ".xlsx"):
        _src = TABLES / (_old + _ext)
        if _src.exists():
            _sh52.copyfile(_src, TABLES / (_new + _ext))
            _src.unlink()
print("Renumbered to the SCS scheme: Table 4, Supp Tables S5, S7, S8")

# Table 4 (v3.0 Tables 6 + 7) and Supp Table S8. Source: the NB06 association caches - the one
# item whose values are not in the published dataset, by design (README section 7).
print("Table 4 (typology-thermal association):")
display(_tbls["Table_6"])

begin("Supp. Table S7")
# Supp Table S7 is the hit-rate table; it must be on the cohort basis
_s7 = _tbls["Supp_Table_7"]
_n7 = sorted(set(_s7[~_s7["_group"]]["No. of points"]))
check("Supp Table S7 - hit rates are on the cohort points",
      len(_s7) == 90, f"{len(_s7)} rows, point counts per city {_n7[:6]}...")
print("Supp Table S7 (hit rate by city and buffer) — preview:")
display(_tbls["Supp_Table_7"].drop(columns=["_group"]).head(3))

#### Table 5 · Best single-element attribution (excerpt)

**Manuscript** 3.5 · **v3.0 reference** `_superseded_v3.0/tables/Table_7_best_element_attribution_excerpt` · **Code** the cell below (display; written by the Table 4 cell)

**Data** NB06 association caches

In [ ]:
begin("Table 5")
print("Table 5 (best single-element attribution, excerpt; written by the Table 4 cell):")
display(_tbls["Table_7"])

#### Fig. 15 · Effect sizes (forest plot)

**Manuscript** 3.5 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_19_forest_plot` · **Code** the cell below

**Data** NB06 association caches

In [ ]:
begin("Fig. 15")
# ── Fig. 10 · Association effect sizes (was v3.0 Figure 19, §2.5) ──────
# The v3.0 forest-plot definition, unchanged. Changed for the SCS rebuild: the
# in-figure title is removed (captions.md carries it) and the output follows the
# new numbering. Source: the NB06 association caches - the one item in the paper
# whose values are not in the published dataset, by design (README section 7).
from matplotlib.patches import Patch as _Patch, FancyBboxPatch as _FBox
from matplotlib.lines import Line2D as _Ln

_SPOT, _PER = ['hot', 'cold'], ['day', 'night']
_ELEMS = ['climate_class', 'geology_class', 'elevation_class']
_ELEM_NAME = {'climate_class': 'Climate Zone', 'geology_class': 'Geology', 'elevation_class': 'Elevation Belt'}
_OUTLBL = {'occurrence': 'Occurrence', 'suhii': 'SUHII', 'area': 'Area (L50)'}
_ALPHA = 0.05
def _pf(x):
    s = str(x).strip()
    return 0.0005 if s.startswith('<') else (float(s) if s.replace('.', '', 1).replace('-', '', 1).isdigit() else 1.0)
def _stars(p): return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))

_pa = pd.read_csv(read_source(RESULTS_06T / 'partA_synthesis_cluster25.csv'))
_paSRC = {'occurrence': {}, 'suhii': {}, 'area': {}}
for _, _r in _pa.iterrows():
    _st, _pe = str(_r['Spot Type']).lower(), str(_r['Period']).lower()
    _paSRC['occurrence'][(_st, _pe)] = {'cramers_v': float(_r['V (occ)']), 'p': _pf(_r['p (occ)'])}
    _paSRC['suhii'][(_st, _pe)] = {'eta2': float(_r['η² (suhii)']), 'p': _pf(_r['p (suhii)'])}
    _paSRC['area'][(_st, _pe)] = {'eta2': float(_r['η² (area)']), 'p': _pf(_r['p (area)'])}
_pbR, _pbH = {}, {}
_name2key = {v: k for k, v in _ELEM_NAME.items()}
for _st in _SPOT:
    for _pe in _PER:
        _pb = pd.read_csv(read_source(RESULTS_06T / f'partB_{_st}_{_pe}.csv'))
        for _, _r in _pb.iterrows():
            _e = _name2key.get(str(_r['Element']).strip())
            if _e is None: continue
            _pbR[(_st, _pe, _e, 'occurrence')] = {'cramers_v': float(_r['V'])}
            _pbR[(_st, _pe, _e, 'suhii')] = {'eta2': float(_r['η²_suhii'])}
            _pbR[(_st, _pe, _e, 'area')] = {'eta2': float(_r['η²_area'])}
            _pbH[(_st, _pe, _e, 'occurrence')] = _pf(_r['p(Holm)_occ'])
            _pbH[(_st, _pe, _e, 'suhii')] = _pf(_r['p(Holm)_suhii'])
            _pbH[(_st, _pe, _e, 'area')] = _pf(_r['p(Holm)_area'])

_TH = {'occurrence': {'s': .10, 'm': .30, 'l': .50, 'fl': .60, 'lbl': "Cramér's V"},
       'suhii': {'s': .01, 'm': .06, 'l': .14, 'fl': .22, 'lbl': 'η²'},
       'area': {'s': .01, 'm': .06, 'l': .14, 'fl': .22, 'lbl': 'η² (L50)'}}
_ZC = {'negligible': '#FFAAAA', 'small': '#FFCC99', 'medium': '#FFEE99', 'large': '#AADDAA'}; _ZA = 0.42
_CFILL = '#A8D8A8'
_EFILL = {'climate_class': '#F4A8A8', 'geology_class': '#A8A8F4', 'elevation_class': '#F4CCA8'}
_EMARK = {'climate_class': 'D', 'geology_class': 's', 'elevation_class': '^'}
_ECS, _ECN, _LWS, _LWN = '#CC1100', '#111111', 2.0, 0.9
_FB, _FT = 10, 11
def _effs(out):
    v = []
    for st in _SPOT:
        for pe in _PER:
            a = _paSRC[out].get((st, pe))
            if a: v.append(a['cramers_v'] if out == 'occurrence' else a['eta2'])
            for e in _ELEMS:
                b = _pbR.get((st, pe, e, out))
                if b: v.append(b['cramers_v'] if out == 'occurrence' else b['eta2'])
    return v

fig19, _axes = plt.subplots(1, 3, figsize=(W_FULL, 6.0), gridspec_kw={'wspace': 0.46})
for _ax, _out in zip(_axes, ['occurrence', 'suhii', 'area']):
    _t = _TH[_out]; _s, _m, _l = _t['s'], _t['m'], _t['l']
    _al = _effs(_out); _xl = max(_t['fl'], max(_al) * 1.15) if _al else _t['fl']
    _ax.set_title(_OUTLBL[_out], fontsize=_FT, fontweight='bold', pad=8)
    _ax.set_xlabel(_t['lbl'], fontsize=_FB); _ax.set_xlim(-0.01, _xl)
    _ax.axvspan(-0.01, _s, color=_ZC['negligible'], alpha=_ZA, zorder=0)
    _ax.axvspan(_s, _m, color=_ZC['small'], alpha=_ZA, zorder=0)
    _ax.axvspan(_m, _l, color=_ZC['medium'], alpha=_ZA, zorder=0)
    _ax.axvspan(_l, _xl, color=_ZC['large'], alpha=_ZA, zorder=0)
    for _rf, _ls in ((_s, '--'), (_m, ':'), (_l, '-.')):
        _ax.axvline(_rf, color='#888888', lw=0.7, ls=_ls, zorder=1)
    _ax.axvline(0, color='#CCCCCC', lw=0.7, zorder=1)
    _yl, _yp, _yc = [], [], 0
    for st in _SPOT:
        for pe in _PER:
            _yl.append(f'{st.capitalize()} / {pe.capitalize()}'); _yp.append(_yc)
            a = _paSRC[_out].get((st, pe))
            if a:
                _eA = a['cramers_v'] if _out == 'occurrence' else a['eta2']; _sA = a['p'] < _ALPHA
                _ax.scatter(_eA, _yc, color=_CFILL, s=110, marker='o', zorder=5,
                            edgecolors=_ECS if _sA else _ECN, linewidths=_LWS if _sA else _LWN)
                if _sA:
                    _ax.text(_eA + _xl * 0.015, _yc + 0.20, _stars(a['p']), fontsize=8,
                             color=_ECS, va='bottom', fontweight='bold')
            _off = {'climate_class': -0.22, 'geology_class': 0.0, 'elevation_class': 0.22}
            for e in _ELEMS:
                b = _pbR.get((st, pe, e, _out))
                if b:
                    _eB = b['cramers_v'] if _out == 'occurrence' else b['eta2']
                    _sB = _pbH.get((st, pe, e, _out), 1.0) < _ALPHA
                    _ax.scatter(_eB, _yc + _off[e], color=_EFILL[e], s=60, marker=_EMARK[e], zorder=4,
                                edgecolors=_ECS if _sB else _ECN, linewidths=_LWS if _sB else _LWN)
            _yc += 1
    _ax.set_yticks(_yp); _ax.set_yticklabels(_yl, fontsize=_FB); _ax.invert_yaxis()
    _ax.grid(True, axis='x', alpha=0.18, ls='--', zorder=2); _ax.tick_params(axis='x', labelsize=_FB - 1)
    _ax.spines['top'].set_visible(False); _ax.spines['right'].set_visible(False)

# (no in-figure title; captions.md carries it)
fig19.subplots_adjust(top=0.91, bottom=0.205, left=0.08, right=0.98)
_BT, _BH = 0.020, 0.130
fig19.add_artist(_FBox((0.01, _BT), 0.97, _BH, boxstyle='square,pad=0', facecolor='#fafafa',
                       edgecolor='#aaaaaa', linewidth=1.2, transform=fig19.transFigure, zorder=0, clip_on=False))
for _sx in (0.34, 0.67):
    fig19.add_artist(_Ln([_sx, _sx], [_BT + 0.006, _BT + _BH - 0.006], transform=fig19.transFigure,
                     color='#cccccc', lw=1.0, zorder=1, clip_on=False))
_lkw = dict(fontsize=_FB - 0.5, frameon=False, handlelength=1.2, handletextpad=0.5, borderpad=0.0,
            columnspacing=0.9, bbox_transform=fig19.transFigure)
_ay = _BT + 0.018
_zh = [_Patch(facecolor=_ZC[k], alpha=_ZA + 0.2, edgecolor='#888', lw=0.6, label=v)
       for k, v in [('negligible', 'Negligible'), ('small', 'Small'), ('medium', 'Medium'), ('large', 'Large')]]
_lg1 = fig19.legend(handles=_zh, title='Effect Size Zone', title_fontsize=_FB, loc='lower left',
                    bbox_to_anchor=(0.03, _ay), ncol=2, **_lkw); _lg1.get_title().set_fontweight('bold')
_nt = '#666666'
_ph = [_Ln([0], [0], marker='o', ls='none', color=_CFILL, markeredgecolor=_nt, markeredgewidth=1.0, markersize=9, label='Composite typology'),
       _Ln([0], [0], marker='D', ls='none', color=_EFILL['climate_class'], markeredgecolor=_nt, markeredgewidth=1.0, markersize=7, label='Climate Zone'),
       _Ln([0], [0], marker='s', ls='none', color=_EFILL['geology_class'], markeredgecolor=_nt, markeredgewidth=1.0, markersize=7, label='Geology'),
       _Ln([0], [0], marker='^', ls='none', color=_EFILL['elevation_class'], markeredgecolor=_nt, markeredgewidth=1.0, markersize=8, label='Elevation Belt')]
_lg2 = fig19.legend(handles=_ph, title='Predictor', title_fontsize=_FB, loc='lower left',
                    bbox_to_anchor=(0.36, _ay), ncol=2, **_lkw); _lg2.get_title().set_fontweight('bold')
_sh = [_Ln([0], [0], marker='o', ls='none', color='#AAAAAA', markeredgecolor=_ECS, markeredgewidth=_LWS, markersize=10, label='Significant'),
       _Ln([0], [0], marker='o', ls='none', color='#AAAAAA', markeredgecolor=_ECN, markeredgewidth=_LWN, markersize=10, label='Not significant')]
_lg3 = fig19.legend(handles=_sh, title='Significance (p < 0.05)', title_fontsize=_FB, loc='lower left',
                    bbox_to_anchor=(0.69, _ay), ncol=1, **_lkw); _lg3.get_title().set_fontweight('bold')
_p19 = savefig(fig19, MAIN_FIG / "Fig_15_association_effect_sizes")
plt.show(); print("Fig. 10 ->", ", ".join(p.name for p in _p19))

## 2 · Supplementary figures

#### S1 · Population trajectories: S1.0 national, S1.1 cities, S1.2 clusters

**Manuscript** 2.1 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_2_population_trend`, `supplementary_figures/1.1_city_trends`, `1.2_cluster_trends` · **Code** the cell below

**Data** WUP table; NB01 city regions and cluster trend figures

In [ ]:
begin("S1")
fig2, stats2 = viz.create_composite_population_trend(
    cities_in_aoi, save_dir=None, figsize=(11.69, 5.845), dpi=300,
    mode="aoi", all_cities_df=wup)

paths = savefig(fig2, MAIN_FIG / "Figure_2_population_trend")
plt.show()
print("Saved:", [p.name for p in paths])
print("Totals — 2024: {pop_2024_k:,.0f}k over {n_aois} AOIs".format(**stats2))


# ── S1.1 · city trajectories ──
CITY_TREND_DIR = SUP_FIG / "1.1_city_trends"
CITY_TREND_DIR.mkdir(exist_ok=True)

years5 = [int(y) for y in YEAR_COLS if int(y) % 5 == 0]
def plot_city_trend(row, save_noext):
    ys = [row[str(y)] for y in years5]            # thousands
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    ax.plot(years5, ys, color="navy", marker="s", ms=3.5, lw=1.2, alpha=0.8)
    for yr, col in ((2024, "crimson"), (2050, "darkgreen")):
        if str(yr) in row.index:
            ax.axvline(yr, color=col, ls="--", lw=0.9)
            ax.scatter(yr, row[str(yr)], marker="D", s=55, color=col,
                       edgecolors="black", lw=0.8, zorder=5)
    ax.set_title(str(row["City_Name"]), fontweight="bold", fontsize=11)
    ax.set_xlabel("Year"); ax.set_ylabel("Population (thousands)")
    ax.grid(alpha=0.25); fig.tight_layout()
    p = savefig(fig, save_noext, formats=("pdf", "svg"))
    plt.close(fig); return p

trend_map = []
for i, (_, row) in enumerate(wup.iterrows(), start=1):
    tag = f"1.1.{i}"
    slug = str(row["City_Name"]).replace(" ", "_").replace("/", "-")
    plot_city_trend(row, CITY_TREND_DIR / f"Supp_Fig_{tag}_{slug}")
    trend_map.append({"supp_fig": tag, "city": row["City_Name"]})
pd.DataFrame(trend_map).to_csv(CITY_TREND_DIR / "_index_1.1_city_trends.csv", index=False)
print(f"Generated {len(trend_map)} city-trend figures  ->  {CITY_TREND_DIR.relative_to(OUT)}")


# ── S1.2 · cluster trends (collected from NB01) ──
CLUS_TREND_DIR = SUP_FIG / "1.2_cluster_trends"
CLUS_TREND_DIR.mkdir(exist_ok=True)
src_dir = RESULTS_AOI / "cluster_trends"

clus_map = []
for i, cid in enumerate(active_clusters, start=1):
    tag = f"1.2.{i}"
    picked = 0
    for ext in ("pdf", "svg"):
        src = src_dir / f"cluster_{cid:02d}_population_trend.{ext}"
        if src.exists():
            shutil.copyfile(read_source(src), CLUS_TREND_DIR / f"Supp_Fig_{tag}_C{cid}.{ext}"); picked += 1
    clus_map.append({"supp_fig": tag, "Cluster25": f"C{cid}", "found_files": picked})
pd.DataFrame(clus_map).to_csv(CLUS_TREND_DIR / "_index_1.2_cluster_trends.csv", index=False)
print(f"Collected {len(clus_map)} cluster-trend figures  ->  {CLUS_TREND_DIR.relative_to(OUT)}")
display(pd.DataFrame(clus_map))

#### S2 · City delineation maps

**Manuscript** 2.3 · **v3.0 reference** `_superseded_v3.0/supplementary_figures/2_city_aoi_maps` · **Code** the cell below

**Data** NB01 three-zone maps

In [ ]:
begin("S2")
AOI_MAP_DIR = SUP_FIG / "2_city_aoi_maps"
AOI_MAP_DIR.mkdir(exist_ok=True)

aoi_map = []
for i, city in enumerate(STUDY_CITIES, start=1):
    tag = f"2.{i}"
    slug = zone_results[city]["city_slug"]
    fig_dir = zone_results[city]["city_dir"] / "figures"
    found = 0
    for ext in ("pdf", "svg"):
        src = fig_dir / f"{slug}_three_zone_map.{ext}"
        if src.exists():
            shutil.copyfile(read_source(src), AOI_MAP_DIR / f"Supp_Fig_{tag}_{slug}.{ext}"); found += 1
    aoi_map.append({"supp_fig": tag, "city": city, "found_files": found})
pd.DataFrame(aoi_map).to_csv(AOI_MAP_DIR / "_index_2_city_aoi_maps.csv", index=False)
print(f"Collected {len(aoi_map)} per-city AOI maps  ->  {AOI_MAP_DIR.relative_to(OUT)}")
display(pd.DataFrame(aoi_map).head())

#### S3 · Stakeholder sample maps: S3.1 cluster map; S3.2 city map

**Manuscript** 2.3 · **v3.0 reference** `_superseded_v3.0/supplementary_figures/3_stakeholder_maps` (Supp 3.1, 3.2; the radial chart is now main Fig. 4) · **Code** the cell below

**Data** now NB02 caches; to be keyed on package `stakeholder_perception.city`

In [ ]:
begin("S3")
STAKE     = ROOT / "Results" / "02_stakeholder_analysis"
STAKE_FIG = STAKE / "Figures"
STAKE_OUT = SUP_FIG / "S3_stakeholder_maps"
STAKE_OUT.mkdir(exist_ok=True)

# ── Supp. Fig. S3.1 / S3.2 — collected as-is from cached NB02 renders ──
_nb02_maps = [
    ("Supp Fig S3.1", "cluster25_stakeholder_map",         "Supp_Fig_S3.1_cluster_stakeholder_map"),
    ("Supp Fig S3.2", "Stakeholder_Distribution_Map_2024", "Supp_Fig_S3.2_city_stakeholder_map"),
]
for label, srcbase, dstbase in _nb02_maps:
    picked = []
    for ext in ("pdf", "svg", "png"):
        src = STAKE_FIG / f"{srcbase}.{ext}"
        if src.exists():
            shutil.copyfile(read_source(src), STAKE_OUT / f"{dstbase}.{ext}"); picked.append(ext)
    print(f"{label:14} <- {srcbase:40} [{', '.join(picked) or 'MISSING'}]")

#### S4 · Long-term trends, complete (clusters, cities, national)

**Manuscript** 3.1 · **v3.0 reference** `_superseded_v3.0/supplementary_figures/4_temporal_trends` · **Code** the cell below

**Data** NB03 trend figures

In [ ]:
begin("S4")
SUP_T = SUP_FIG / "4_temporal_trends"; SUP_T.mkdir(exist_ok=True)
CL_T  = SUP_T / "4.1_cluster"; CL_T.mkdir(exist_ok=True)
CT_T  = SUP_T / "4.2_city";    CT_T.mkdir(exist_ok=True)

# ── Supp 4.1 — 16 cluster combined charts (collected) ──
_ci = []
for i, cid in enumerate(active_clusters, start=1):
    sd = RESULTS_03 / "Cluster25_Composites" / f"Cluster25_{cid:02d}" / "figures"
    for ext in ("pdf", "svg"):
        s = sd / f"Cluster25_{cid:02d}_temporal_trend.{ext}"
        if s.exists():
            shutil.copyfile(read_source(s), CL_T / f"Supp_Fig_4.1.{i}_C{cid}.{ext}")
    _ci.append({"supp_fig": f"4.1.{i}", "Cluster25": f"C{cid}"})
pd.DataFrame(_ci).to_csv(CL_T / "_index_4.1_cluster.csv", index=False)

# ── Supp 4.2 — 56 city combined charts (collected) ──
_cy = []
for i, city in enumerate(STUDY_CITIES, start=1):
    slug = zone_results[city]["city_slug"]; sd = RESULTS_03 / slug / "figures"
    found = 0
    for ext in ("pdf", "svg"):
        s = sd / f"{slug}_temporal_trend.{ext}"
        if s.exists():
            shutil.copyfile(read_source(s), CT_T / f"Supp_Fig_4.2.{i}_{slug}.{ext}"); found += 1
    _cy.append({"supp_fig": f"4.2.{i}", "city": city, "found_files": found})
pd.DataFrame(_cy).to_csv(CT_T / "_index_4.2_city.csv", index=False)

# ── Supp 4.3 — national composite (collected) ──
_ncf = RESULTS_03 / "National_Composite" / "figures"
for ext in ("pdf", "svg"):
    s = _ncf / f"national_composite_temporal_trend.{ext}"
    if s.exists():
        shutil.copyfile(read_source(s), SUP_T / f"Supp_Fig_4.3_national.{ext}")

print(f"Collected combined trends -> Supp 4.1 (16 clusters), 4.2 ({len(_cy)} cities), 4.3 (national)")
print("MAIN-TEXT MAPPING: Fig 7 & Fig 8 both point to Supp 4.1 (cluster) / 4.2 (city) / 4.3 (national).")

#### S5 · Daily series and persistent heat events

**Manuscript** 3.1 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_10_daily_phe_panel`, `supplementary_figures/7_daily_phe` · **Code** the cell below

**Data** NB04 daily series and charts

In [ ]:
begin("S5")
import matplotlib.dates as _mdates
from matplotlib.gridspec import GridSpecFromSubplotSpec as _GSS
from daily_analysis_utils import run_city_event_pipeline

RAW04 = RESULTS_04 / "raw_extractions"; CLC04 = RESULTS_04 / "cluster_composites"
_YR = STUDY_YEAR
_TIER_C = {"EXCELLENT": "#1a9850", "GOOD": "#f77f00", "FAIR": "#999999", "ACCEPTABLE": "#999999"}

def _period_df(comp, period):
    col = f"{period}_lst_median"
    if comp is None or col not in comp.columns:
        return None
    d = comp[["date", col]].rename(columns={col: "lst_median"}).copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    return d.dropna(subset=["date", "lst_median"]).sort_values("date").reset_index(drop=True)

def _audits(mdf, edf, name):
    return (run_city_event_pipeline(_period_df(mdf, "day"),   _period_df(edf, "day"),   name, "day"),
            run_city_event_pipeline(_period_df(mdf, "night"), _period_df(edf, "night"), name, "night"))

def _national_daily(ds):
    frames = []
    for p in RAW04.glob(f"*_{ds}_UC_daily_{_YR}.csv"):
        d = pd.read_csv(read_source(p)); d["date"] = pd.to_datetime(d["date"], errors="coerce"); frames.append(d)
    allc = pd.concat(frames)
    num = [c for c in allc.columns if c.startswith(("day_lst", "night_lst"))]
    return allc.groupby("date", as_index=False)[num].mean()

def _daily_panel(ax, df, ds_label, ad, an, ds_type, level_lbl=None, fixed_ylim=None):
    DAY_C, DAY_F = "#d73027", "#fdae61"; NGT_C, NGT_F = "#1C1C1C", "#9a9a9a"
    SEL_D, SEL_N = "#a50026", "#000000"
    if df is None:
        ax.text(0.5, 0.5, f"No {ds_label}", transform=ax.transAxes, ha="center", va="center", color="grey"); return
    df = df.copy(); df["date"] = pd.to_datetime(df["date"]); df = df.sort_values("date")
    if {"day_lst_min", "day_lst_max"} <= set(df.columns):
        ax.fill_between(df["date"], df["day_lst_min"], df["day_lst_max"], color=DAY_F, alpha=.35, lw=0, label="Day range")
    ax.plot(df["date"], df["day_lst_median"], color=DAY_C, lw=1.0, label="Day median", zorder=3)
    if {"night_lst_min", "night_lst_max"} <= set(df.columns):
        ax.fill_between(df["date"], df["night_lst_min"], df["night_lst_max"], color=NGT_F, alpha=.45, lw=0, label="Night range")
    ax.plot(df["date"], df["night_lst_median"], color=NGT_C, lw=1.0, label="Night median", zorder=3)
    key = f"{ds_type}_p90_celsius"
    td = ad.get("thresholds", {}).get(key); tn = an.get("thresholds", {}).get(key)
    if td is not None:
        ax.axhline(td, color=DAY_C, ls="--", lw=.9, alpha=.7, label=f"Day P90 {td:.1f}°", zorder=4)
        h = df[df["day_lst_median"] >= td]; ax.scatter(h["date"], h["day_lst_median"], color=DAY_C, s=9, zorder=5, alpha=.6)
    if tn is not None:
        ax.axhline(tn, color=NGT_C, ls="--", lw=.9, alpha=.7, label=f"Night P90 {tn:.1f}°", zorder=4)
        h = df[df["night_lst_median"] >= tn]; ax.scatter(h["date"], h["night_lst_median"], color=NGT_C, s=9, zorder=5, alpha=.6)
    seen = set()
    for e in ad.get(f"{ds_type}_persistent_events", []) + an.get(f"{ds_type}_persistent_events", []):
        k = (e.get("start_date"), e.get("end_date"))
        if k in seen: continue
        seen.add(k)
        try:
            ax.axvspan(pd.to_datetime(e["start_date"]), pd.to_datetime(e["end_date"]) + pd.Timedelta(days=1),
                       color="grey", alpha=.12, lw=0, zorder=1)
        except Exception: pass
    # ── fix the y-scale FIRST so the selection bracket sits correctly ──
    if fixed_ylim is not None:
        ax.set_ylim(*fixed_ylim)
    else:
        lo = float(np.nanmin(df["night_lst_min"] if "night_lst_min" in df else df["night_lst_median"]))
        hi = float(np.nanmax(df["day_lst_max"] if "day_lst_max" in df else df["day_lst_median"]))
        ax.set_ylim(lo - 2, hi + 2)
    y0, y1 = ax.get_ylim(); gap = (y1 - y0) * .09
    def _br(sd, ed, col, above, tier=None):
        """Draw the selected-event bracket (⊢——⊣) and annotate its tier below it."""
        try: s, e = pd.to_datetime(sd), pd.to_datetime(ed)
        except Exception: return
        bh = (y1 - y0) * .025; b = above - gap
        ax.plot([s, e], [b, b], color=col, lw=2.0, zorder=8)
        ax.plot([s, s], [b - bh, b + bh], color=col, lw=2.0, zorder=9); ax.plot([e, e], [b - bh, b + bh], color=col, lw=2.0, zorder=9)
        if tier:
            ax.annotate(tier, xy=(s + (e - s) / 2, b - bh), xytext=(0, -1.5),
                        textcoords="offset points", ha="center", va="top",
                        fontsize=4.8, fontweight="bold", color=_TIER_C.get(tier, col), zorder=10)
    sd, sn = ad.get("selected_event"), an.get("selected_event")
    if sd and td is not None: _br(sd["start_date"], sd["end_date"], SEL_D, td, sd.get("tier", "FAIR"))
    if sn and tn is not None: _br(sn["start_date"], sn["end_date"], SEL_N, tn, sn.get("tier", "FAIR"))
    ax.set_ylabel((f"{level_lbl}\n" if level_lbl else "") + f"{ds_label} (°C)", fontsize=6.5)
    ax.tick_params(labelsize=5.5); ax.grid(axis="y", ls=":", lw=.4, alpha=.5)
    # legend — no per-tier entries (tier is annotated under each selection bracket)
    hnd, lab = ax.get_legend_handles_labels()
    ax.legend(hnd, lab, loc="upper right", fontsize=5.0, ncol=3, framealpha=.85,
              handlelength=1.1, columnspacing=.7, borderaxespad=.2)

_LG_M = pd.read_csv(read_source(RAW04 / f"Lagos_MODIS_UC_daily_{_YR}.csv")); _LG_E = pd.read_csv(read_source(RAW04 / f"Lagos_ERA5_UC_daily_{_YR}.csv"))
_C13_M = pd.read_csv(read_source(CLC04 / f"Cluster13_MODIS_composite_{_YR}.csv")); _C13_E = pd.read_csv(read_source(CLC04 / f"Cluster13_ERA5_composite_{_YR}.csv"))
_NAT_M = _national_daily("MODIS"); _NAT_E = _national_daily("ERA5")
_LEVELS = [("Lagos (city)", _LG_M, _LG_E), ("Cluster 13", _C13_M, _C13_E), ("National", _NAT_M, _NAT_E)]

# ── uniform temperature scale across ALL six sub-panels ──
def _series_range(d):
    lo = float(np.nanmin(d["night_lst_min"] if "night_lst_min" in d else d["night_lst_median"]))
    hi = float(np.nanmax(d["day_lst_max"] if "day_lst_max" in d else d["day_lst_median"]))
    return lo, hi
_ranges = [_series_range(d) for d in (_LG_M, _LG_E, _C13_M, _C13_E, _NAT_M, _NAT_E) if d is not None]
_GYLIM = (min(r[0] for r in _ranges) - 2, max(r[1] for r in _ranges) + 2)

fig10 = plt.figure(figsize=(8.27, 9.6))
_outer = fig10.add_gridspec(3, 1, hspace=0.14, top=0.985, bottom=0.05, left=0.10, right=0.97)
for _i, (_lbl, _m, _e) in enumerate(_LEVELS):
    _ad, _an = _audits(_m, _e, _lbl)
    _in = _GSS(2, 1, subplot_spec=_outer[_i], hspace=0.06)
    _axm = fig10.add_subplot(_in[0]); _axe = fig10.add_subplot(_in[1], sharex=_axm)
    _daily_panel(_axm, _m, "MODIS", _ad, _an, "modis", level_lbl=_lbl, fixed_ylim=_GYLIM)
    _daily_panel(_axe, _e, "ERA5-Land", _ad, _an, "era5", fixed_ylim=_GYLIM)
    _axm.tick_params(labelbottom=False)
    _axe.xaxis.set_major_locator(_mdates.MonthLocator()); _axe.xaxis.set_major_formatter(_mdates.DateFormatter("%b"))
    _axe.tick_params(axis="x", labelsize=5.5)
paths = savefig(fig10, MAIN_FIG / "Figure_10_daily_phe_panel")
plt.show(); print("Figure 10:", [p.name for p in paths], "| uniform y =", tuple(round(v,1) for v in _GYLIM))


# ── Supp 7.1 (all cities) / 7.2 (all clusters) — collected daily charts (original format) ──
SUP7 = SUP_FIG / "7_daily_phe"; SUP7.mkdir(exist_ok=True)
S71 = SUP7 / "7.1_city"; S71.mkdir(exist_ok=True); S72 = SUP7 / "7.2_cluster"; S72.mkdir(exist_ok=True)
_CDT = RESULTS_04 / "charts" / "city_daily_timeseries"
_CLD = RESULTS_04 / "charts" / "cluster_daily_composites"
_n71 = 0
for i, city in enumerate(STUDY_CITIES, start=1):
    slug = zone_results[city]["city_slug"]
    for ext in ("svg", "pdf"):
        # NB04 named a few multi-word cities with spaces rather than underscores
        for _base in (slug, slug.replace("_", " ")):
            s = _CDT / f"{_base}_daily_LST_{_YR}.{ext}"
            if s.exists():
                shutil.copyfile(read_source(s), S71 / f"Supp_Fig_7.1.{i}_{slug}.{ext}"); _n71 += (ext == "svg"); break
_n72 = 0
for i, cid in enumerate(active_clusters, start=1):
    for ext in ("svg", "pdf"):
        s = _CLD / f"Cluster{cid:02d}_daily_LST_{_YR}.{ext}"
        if s.exists(): shutil.copyfile(read_source(s), S72 / f"Supp_Fig_7.2.{i}_C{cid}.{ext}"); _n72 += (ext == "svg")
print(f"Supp 7.1 collected {_n71} city charts; Supp 7.2 collected {_n72} cluster charts (original format)")

#### S6 · Event frequency by cluster, day / night

**Manuscript** 3.1 · **v3.0 reference** `_superseded_v3.0/supplementary_figures/8_event_frequency` · **Code** the cell below

**Data** NB04 charts

In [ ]:
begin("S6")
# ── S6 · Event frequency by cluster, day / night (collected; v3.0 Supp Fig 8.1 / 8.2) ──
_CHARTS = RESULTS_04 / "charts"
for label, s, dst, base in [
    ("Supp Fig 8.1", _CHARTS / "Cluster_event_frequency_day.svg",   SUP_FIG / "8_event_frequency", "Supp_Fig_8.1_cluster_event_frequency_day"),
    ("Supp Fig 8.2", _CHARTS / "Cluster_event_frequency_night.svg", SUP_FIG / "8_event_frequency", "Supp_Fig_8.2_cluster_event_frequency_night"),
]:
    dst.mkdir(exist_ok=True)
    picked = []
    for ext in ("svg", "pdf"):
        sf = s.with_suffix(f".{ext}")
        if sf.exists(): shutil.copyfile(read_source(sf), dst / f"{base}.{ext}"); picked.append(ext)
    print(f"{label:12} <- {s.name:44} [{', '.join(picked) or 'MISSING'}]")

#### S7 · Agreement between thermal products: S7.1 MODIS vs ERA5-Land; S7.2 SEVIRI vs ERA5-Land

**Manuscript** 3.1 · **v3.0 reference** `_superseded_v3.0/supplementary_figures/9_hexbin`; S7.2 has no v3.0 file (NB04's SEVIRI_vs_ERA5_event_window_correlation is the reference)` · **Code** the cell below

**Data** NB04 daily series; S7.2 to come from package `city_year_summary`

In [ ]:
begin("S7")
# ── Supp Fig 9 — ERA5-Land vs MODIS daily hexbin, all cities pooled (title uses '-') ──
from src.visualization import create_era5_modis_hexbin
from src.daily_analysis_utils import load_daily_csv, compute_correlation_metrics
_RAW09 = RESULTS_04 / "raw_extractions"
_cities09 = pd.read_csv(read_source(RESULTS_04 / "city_heat_event_summary.csv"))["city"].tolist()
_pooled09 = {"day": {"era5": [], "modis": []}, "night": {"era5": [], "modis": []}}
for _city in _cities09:
    _mcsv = _RAW09 / f"{_city}_MODIS_UC_daily_{STUDY_YEAR}.csv"
    _ecsv = _RAW09 / f"{_city}_ERA5_UC_daily_{STUDY_YEAR}.csv"
    if not (_mcsv.exists() and _ecsv.exists()):
        continue
    for _p in ("day", "night"):
        _m = load_daily_csv(read_source(_mcsv), _p); _e = load_daily_csv(read_source(_ecsv), _p)
        if _m is None or _e is None:
            continue
        _mg = pd.merge(
            _m[["date", "lst_median"]].rename(columns={"lst_median": "modis"}),
            _e[["date", "lst_median"]].rename(columns={"lst_median": "era5"}),
            on="date", how="inner").dropna()
        if _mg.empty:
            continue
        _pooled09[_p]["era5"]  += _mg["era5"].tolist()
        _pooled09[_p]["modis"] += _mg["modis"].tolist()
for _p in ("day", "night"):
    _pooled09[_p]["metrics"] = compute_correlation_metrics(
        pd.Series(_pooled09[_p]["modis"]), pd.Series(_pooled09[_p]["era5"]))
_S9DIR = SUP_FIG / "9_hexbin"; _S9DIR.mkdir(exist_ok=True)
fig9 = create_era5_modis_hexbin(
    paired_data = _pooled09,
    title       = f"All Cities Pooled - ERA5-Land vs MODIS ({STUDY_YEAR})",
    output_path = str(_S9DIR / "Supp_Fig_9_era5_vs_modis_hexbin.svg"),
    dpi         = config.FIGURE_DPI,
    show_inline = True,
)
if fig9 is not None:      # emit a PDF sibling (function itself writes only the SVG)
    fig9.savefig(str(_S9DIR / "Supp_Fig_9_era5_vs_modis_hexbin.pdf"),
                 format="pdf", bbox_inches="tight", facecolor="white")
print("Supp Fig 9 -> Supp_Fig_9_era5_vs_modis_hexbin.(svg|pdf)  "
      f"[day n={_pooled09['day']['metrics'].get('n')}, night n={_pooled09['night']['metrics'].get('n')}]")

#### S8 · Gi* z-score distributions (S8.0 Uyo; all cities)

**Manuscript** 3.2 · **v3.0 reference** `_superseded_v3.0/main_figures/Figure_14_uyo_gistar_dist`, `supplementary_figures/10_gistar_distributions` · **Code** the cell below

**Data** NB05 figures

In [ ]:
begin("S8")
# ── Figure 14 — Uyo Gi* Z-score distribution ──
_S10 = SUP_FIG / "10_gistar_distributions"; _S10.mkdir(exist_ok=True)
p14 = _copy_pair("Uyo_gistar_dist", MAIN_FIG, "Figure_14_uyo_gistar_dist")
print("Figure 14 (Uyo) ->", p14 or "MISSING")

# ── Supp Fig 10.1–10.55 — all other cities' Gi* distributions (alphabetical, Uyo excluded) ──
_gis = sorted(p.stem.replace("_gistar_dist", "") for p in _F05.glob("*_gistar_dist.svg"))
_others = [c for c in _gis if c != "Uyo"]
_n10 = 0
for i, city in enumerate(_others, start=1):
    got = _copy_pair(f"{city}_gistar_dist", _S10, f"Supp_Fig_10.{i}_{city.replace(' ', '_')}")
    _n10 += ("svg" in got)
print(f"Supp Fig 10: collected {_n10} city Gi* distributions (of {len(_others)} non-Uyo cities)")

if _REPAIRED:
    print("  Repaired %d collected .pdf file(s) that held SVG content "
          "(an NB05 saving defect; the source notebook is untouched)."
          % len(_REPAIRED))

#### S9 · City spot area, ranked (hot, cold)

**Manuscript** 3.2 · **v3.0 reference** `_superseded_v3.0/supplementary_figures/11_city_area_ranked` · **Code** the cell below

**Data** NB05 figures; areas to come from the package

In [ ]:
begin("S9")
# ── Supp Fig 11.1 / 11.2 — city hot-/cold-spot area ranked (collected) ──
_S11 = SUP_FIG / "11_city_area_ranked"; _S11.mkdir(exist_ok=True)
for src, dst in [("city_hotspot_area_ranked",  "Supp_Fig_11.1_city_hotspot_area_ranked"),
                 ("city_coldspot_area_ranked", "Supp_Fig_11.2_city_coldspot_area_ranked")]:
    print(f"{dst:44} <- {src:30} [{', '.join(_copy_pair(src, _S11, dst)) or 'MISSING'}]")

#### S10 · Population within spots, full maps and half-donuts

**Manuscript** 3.3 · **v3.0 reference** `_superseded_v3.0/supplementary_figures/12_population_exposure` · **Code** the cell below

**Data** NB05 figures

In [ ]:
begin("S10")
_S12 = SUP_FIG / "12_population_exposure"; _S12.mkdir(exist_ok=True)
def _copy12(stem_src, base_dst):
    got = []
    for ext in ("svg", "pdf"):
        s = _F05 / f"{stem_src}.{ext}"
        if s.exists(): shutil.copyfile(read_source(s), _S12 / f"{base_dst}.{ext}"); got.append(ext)
    return got

_copy12("national_hot_summary_map",  "Supp_Fig_12.1.1_national_hotspot_exposure_map")
_copy12("national_cold_summary_map", "Supp_Fig_12.1.2_national_coldspot_exposure_map")
_copy12("national_population_halfdonut", "Supp_Fig_12.4_national_halfdonut")
_clids12 = sorted(int(p.stem.replace("cluster", "").split("_")[0]) for p in _F05.glob("cluster*_population_halfdonut.svg"))
for _i, _cl in enumerate(_clids12, start=1):
    _copy12(f"cluster{_cl}_population_halfdonut", f"Supp_Fig_12.2.{_i}_Cluster{_cl}_halfdonut")
_cities12 = sorted(p.stem.replace("_population_halfdonut", "") for p in _F05.glob("*_population_halfdonut.svg")
                   if not p.stem.startswith(("cluster", "national")))
for _i, _city in enumerate(_cities12, start=1):
    _copy12(f"{_city}_population_halfdonut", f"Supp_Fig_12.3.{_i}_{_city.replace(' ', '_')}_halfdonut")
print(f"Supp Fig 12: 2 maps + national half-donut + {len(_clids12)} clusters + {len(_cities12)} cities collected")

#### S11 · City-level stakeholder perception

**Manuscript** 3.4 · **v3.0 reference** `_superseded_v3.0/supplementary_figures/6_city_perception` · **Code** the cell below

**Data** NB03 perception panel

In [ ]:
begin("S11")
# Supp. Fig. S12 — NB03's city-level panel, collected unchanged
PERC = RESULTS_03 / "Perception_Panels"
_S12 = SUP_FIG / "S12_city_perception"; _S12.mkdir(parents=True, exist_ok=True)
_picked = []
for _ext in ("pdf", "svg", "png"):
    _s = PERC / f"Perception_City_3panel.{_ext}"
    if _s.exists():
        shutil.copyfile(read_source(_s), _S12 / f"Supp_Fig_S12_city_perception.{_ext}")
        _picked.append(_ext)
print("Supp Fig S12 <- Perception_City_3panel  [%s]" % (", ".join(_picked) or "MISSING"))

## 3 · Supplementary tables

#### Supp. Table S1 · The 24 physiographic landscape typologies

**Manuscript** 2.1 · **v3.0 reference** `_superseded_v3.0/appendices/Appendix_1_PLT_reference` · **Code** the cell below

**Data** typology map

In [ ]:
begin("Supp. Table S1")
# ── National Cluster25 (PLT) reference — matches manuscript Appendix I ──
# Columns: S/No | Colour | Cluster (CL-X) | Climate Class | Geology Class | Elevation Class.
def _predom(col):
    a = (landscape_gdf.groupby(["Cluster25", col])["area_sqkm"].sum().reset_index()
         .sort_values(["Cluster25", "area_sqkm"], ascending=[True, False]))
    return a.groupby("Cluster25").first()[col]

_nat = pd.DataFrame({
    "Climate Class":   _predom("Climate_Class"),
    "Geology Class":   _predom("Geology_Class"),
    "Elevation Class": _predom("Elevation_Class"),
}).reset_index()
_nat["Cluster25"] = _nat["Cluster25"].astype(int)
_nat = _nat.sort_values("Cluster25").reset_index(drop=True)
_nat["Cluster"] = _nat["Cluster25"].map(lambda c: f"CL-{c}")
_nat["Colour"]  = _nat["Cluster25"].map(lambda c: config.CLUSTER25_COLORS.get(int(c), "#CCCCCC"))
_nat.insert(0, "S/No", range(1, len(_nat) + 1))

# CSV carries the hex under "Colour"; the inline preview and Excel render it as a swatch.
appx = _nat[["S/No", "Cluster", "Climate Class", "Geology Class", "Elevation Class", "Colour"]]

_styler = (_nat[["S/No", "Colour", "Cluster", "Climate Class", "Geology Class", "Elevation Class"]]
           .style
           .set_caption("Appendix I — Physiographic Landscape Typologies in Nigeria and their internal makeup")
           .set_table_styles([
               {"selector": "th", "props": [("background-color", "#404040"),
                                            ("color", "white"), ("font-weight", "bold")]},
               {"selector": "caption", "props": [("font-weight", "bold"),
                                                 ("font-size", "13px"), ("text-align", "left")]},
           ])
           .set_properties(**{"font-size": "11px"})
           .apply(lambda s: [f"background-color:{v}; color:{v}; border:1px solid #999" for v in s],
                  subset=["Colour"], axis=0)
           .hide(axis="index"))
display(_styler)

# ── (a) CSV (Colour column carries the hex) ──
csv_p = APPENDIX / "Appendix_1_PLT_reference.csv"
appx.to_csv(csv_p, index=False)

# ── (b) Excel — Colour rendered as a filled swatch (no text) ──
xlsx_p = APPENDIX / "Appendix_1_PLT_reference.xlsx"
wb = xlsxwriter.Workbook(str(xlsx_p))
ws = wb.add_worksheet("Appendix I")
hdr  = wb.add_format({"bold": True, "bg_color": "#404040", "font_color": "white",
                      "border": 1, "align": "center", "valign": "vcenter", "text_wrap": True})
cell = wb.add_format({"border": 1, "valign": "vcenter", "text_wrap": True})
cols = ["S/No", "Colour", "Cluster", "Climate Class", "Geology Class", "Elevation Class"]
for j, c in enumerate(cols):
    ws.write(0, j, c, hdr)
for i, (_, r) in enumerate(_nat.iterrows(), start=1):
    hx = str(r["Colour"])
    ws.write(i, 0, int(r["S/No"]), cell)
    ws.write(i, 1, "", wb.add_format({"bg_color": hx, "border": 1}))
    ws.write(i, 2, r["Cluster"], cell)
    ws.write(i, 3, r["Climate Class"], cell)
    ws.write(i, 4, r["Geology Class"], cell)
    ws.write(i, 5, r["Elevation Class"], cell)
ws.set_column(0, 0, 6); ws.set_column(1, 1, 9); ws.set_column(2, 2, 9); ws.set_column(3, 5, 30)
ws.set_row(0, 26)
wb.close()
print(f"Saved {len(appx)}-cluster Appendix I (manuscript format):", csv_p.name, "+", xlsx_p.name)

#### Supp. Table S2 · Normality tests (Shapiro–Wilk) for the association inputs

**Manuscript** 2.3 · **v3.0 reference** none (built in the format of the other supplementary tables) · **Code** none yet

**Data** NB06 normality output

> No cell yet: built in its step.

#### Supp. Table S3 · Heat events per city

**Manuscript** 3.1 · **v3.0 reference** `_superseded_v3.0/tables/Supp_Table_4_event_frequency` · **Code** the cell below

**Data** now NB04 event summary; to come from package `city_year_summary`

In [ ]:
begin("Supp. Table S3")
# ── Supp Table 4 — event-frequency breakdown (city / cluster / national; ERA5 PHE basis) ──
_ce = pd.read_csv(read_source(RESULTS_04 / "city_heat_event_summary.csv"))
_stats = ["n_persistent_events", "total_persistent_days", "longest_event_days", "mean_event_duration_days"]
_ecols = [f"era5_{p}_{s}" for p in ("day", "night") for s in _stats]
def _t4_row(level, entity, cl, row):
    d = {"Level": level, "Entity": entity, "Cluster25": (f"C{int(cl)}" if pd.notna(cl) else "")}
    for p in ("day", "night"):
        for s, nm in zip(_stats, ["Events", "PersistDays", "Longest", "MeanDur"]):
            d[f"{p.title()}_{nm}"] = round(float(row[f"era5_{p}_{s}"]), 1)
    return d
_rows4 = [_t4_row("City", r["city"], r["cluster25"], r) for _, r in _ce.iterrows()]
for cl, g in _ce.groupby("cluster25"):
    _rows4.append(_t4_row("Cluster", f"Cluster {int(cl)}", cl, g[_ecols].mean()))
_rows4.append(_t4_row("National", "National", float("nan"), _ce[_ecols].mean()))
supp4 = pd.DataFrame(_rows4)
csvS4 = TABLES / "Supp_Table_4_event_frequency.csv"; supp4.to_csv(csvS4, index=False)
print(f"Supp Table 4: {len(supp4)} rows -> {csvS4.name}")
display(supp4.head(4))

#### Supp. Table S4 · Selected events, event tier (day / night) and sensor agreement

**Manuscript** 3.1 · **v3.0 reference** `_superseded_v3.0/tables/Supp_Table_5_selected_events_correlations` · **Code** the cell below

**Data** as Supp. Table S3

In [ ]:
begin("Supp. Table S4")
# ── Supp Table 5 — selected-event windows, tiers, correlations (per city) ──
_ren = {"city": "City", "cluster25": "Cluster25", "validation_status": "Overall_Tier",
        "day_sel_start": "Day_Start", "day_sel_end": "Day_End", "day_sel_duration_days": "Day_Days", "day_sel_tier": "Day_Tier",
        "day_corr_r": "Day_r", "day_corr_r2": "Day_r2", "day_corr_bias": "Day_Bias", "day_corr_rmse": "Day_RMSE",
        "night_sel_start": "Night_Start", "night_sel_end": "Night_End", "night_sel_duration_days": "Night_Days", "night_sel_tier": "Night_Tier",
        "night_corr_r": "Night_r", "night_corr_r2": "Night_r2", "night_corr_bias": "Night_Bias", "night_corr_rmse": "Night_RMSE"}
supp5 = _ce[[c for c in _ren if c in _ce.columns]].rename(columns=_ren)
supp5["Cluster25"] = supp5["Cluster25"].map(lambda c: f"C{int(c)}")
csvS5 = TABLES / "Supp_Table_5_selected_events_correlations.csv"; supp5.to_csv(csvS5, index=False)
print(f"Supp Table 5: {len(supp5)} rows -> {csvS5.name}")
display(supp5.head(4))

#### Supp. Table S5 · Spots and contours per city

**Manuscript** 3.2 · **v3.0 reference** `_superseded_v3.0/tables/Supp_Table_6_city_zone_footprints` · **Code** the Table 2 cell (`build_hotspot_tables`)

**Data** now NB05 summary; to come from the package

#### Supp. Table S6 · Stakeholder accounts per city

**Manuscript** 3.4 · **v3.0 reference** `_superseded_v3.0/tables/Supp_Table_3_corroboration_complete` · **Code** the cell below

**Data** NB01 lookup, NB03 trend tables and agreement table; trends to come from package `trend_statistics`

In [ ]:
begin("Supp. Table S6")
# ── Supp. Table S6 · Stakeholder accounts per city (was v3.0 Supp Table 3) ──
# Unchanged content: every city and cluster, including those with no stakeholders,
# each with its own trend. Renumbered, and no longer cut down to Table 3 columns.
# ── Supp Table 3 — complete corroboration (all 56 cities + 16 clusters + national) ──
_lk = pd.read_csv(read_source(RESULTS_AOI / "city_cluster25_lookup.csv"))
_city_cl = {str(r["city_name"]).strip(): int(r["Cluster25"]) for _, r in _lk.iterrows()}

def _uc_trend(stats_df, period):
    s = stats_df[(stats_df["zone"] == "urban_core") & (stats_df["period"] == period)]
    if s.empty: return ""
    return "Significant increase" if "increas" in str(s["mk_trend"].iloc[0]).lower() else "No significant trend"

# all-city rows with UC day/night trend from per-city annual stats
_city_rows = []
for city in STUDY_CITIES:
    slug = zone_results[city]["city_slug"]
    ap = RESULTS_03 / slug / f"{slug}_annual_lst.csv"
    if not ap.exists():
        continue
    a = pd.read_csv(read_source(ap))
    cid = _city_cl.get(city)
    _city_rows.append({"Level": "City", "Unit": city,
                       "Cluster25": (f"C{cid:02d}" if cid else "C*"),
                       "LST_Day_Trend": _uc_trend(a, "day"), "LST_Night_Trend": _uc_trend(a, "night")})
_allcity = pd.DataFrame(_city_rows)

# merge represented stakeholder columns onto the full city list
_repr_cols = [c for c in comp.columns if c not in ("Level", "LST_Day_Trend", "LST_Night_Trend", "Cluster25")]
_full_city = _allcity.merge(comp[comp["Level"] == "City"][_repr_cols], on="Unit", how="left")
_full_city["N"] = _full_city["N"].fillna(0).astype(int)

# all 16 clusters with cluster UC trend; represented ones carry stakeholder data
_ast = pd.read_csv(read_source(STATS03 / "Cluster25_Annual_Stats.csv"))
_cl_rows = []
for cid in active_clusters:
    a = _ast[_ast["city"] == f"Cluster25_{cid:02d}"]
    _cl_rows.append({"Level": "Cluster", "Unit": f"Cluster {cid}", "Cluster25": f"C{cid:02d}",
                     "LST_Day_Trend": _uc_trend(a, "day"), "LST_Night_Trend": _uc_trend(a, "night")})
_allcl = pd.DataFrame(_cl_rows)
_repr_cl = comp[comp["Level"] == "Cluster"].copy()
_repr_cl["Cluster25"] = _repr_cl["Cluster25"].str.replace("C0", "C", regex=False)
_allcl["_k"] = _allcl["Cluster25"].str.replace("C0", "C", regex=False)
_full_cl = _allcl.merge(_repr_cl[[c for c in comp.columns if c not in ("Level", "Unit", "LST_Day_Trend", "LST_Night_Trend")]]
                        .assign(_k=lambda d: d["Cluster25"].str.replace("C0", "C", regex=False)).drop(columns="Cluster25"),
                        on="_k", how="left").drop(columns="_k")
_full_cl["N"] = _full_cl["N"].fillna(0).astype(int)

supp3 = pd.concat([_full_city, _full_cl, comp[comp["Level"] == "National"]], ignore_index=True)
# Table 3's cluster rows now carry different columns, so the city table keeps the
# comprehensive column set it was built with rather than being cut down to them.
supp3 = supp3[[c for c in comp.columns if c in supp3.columns]]
csvS3 = TABLES / "Supp_Table_S6_stakeholder_per_city.csv"
supp3.to_csv(csvS3, index=False)
print(f"Supp Table S6: {len(supp3)} rows ({(supp3['N']==0).sum()} without stakeholders) -> {csvS3.name}")
display(supp3[supp3["N"] == 0].head(6))

#### Supp. Table S7 · Hit rate by city and buffer

**Manuscript** 3.4 · **v3.0 reference** `_superseded_v3.0/tables/Supp_Table_7_buffer_sensitivity` · **Code** the Table 2 cell (`build_hotspot_tables`)

**Data** package `hitrate_sensitivity`

#### Supp. Table S8 · Best-element attribution, complete

**Manuscript** 3.5 · **v3.0 reference** `_superseded_v3.0/tables/Supp_Table_8_best_element_attribution_full` · **Code** the Table 2 cell (`build_hotspot_tables`)

**Data** NB06 association caches

## 4 · Assembly

`captions.md` · the main and supplementary contact sheets · print sizes (reported only) · file-format
checks · the old-to-new number map (`number_map.csv`) · the provenance summary (`_data_sources.csv`) ·
the scan of this notebook's code for forbidden data paths · the side-by-side previews and the generated
part of `_preview/INDEX.md` · the assertion summary.

In [ ]:
begin("Assembly")
# ── captions.md, the contact sheet and the assertion summary ───────────
# Captions are generated here so their counts stay tied to the package. Figures carry
# no titles of their own; these captions carry them.
import fitz as _fitzC

_profC = pkg("city_profile")
_pcC = pkg("stakeholder_perception")
_exC = pkg("city_exposure_by_contour")
_spC = pkg("spots")
_perCityC = _pcC.groupby("city").size()
_singlesC = sorted(_perCityC[_perCityC == 1].index)
_multiC = sorted(_perCityC[_perCityC >= 2].index)
_droppedC = sorted(set(_pcC.city) - set(_multiC))
_hotNightC = _exC[(_exC.spot_type == "hot") & (_exC.period == "night")].pop_l50_persons.sum()
_uc13C = _profC[_profC.cluster25 == 13].uc_area_km2.sum()
_share13C = 100.0 * _uc13C / _profC.uc_area_km2.sum()

PERCEPTION_BASE = ("105 participants in 21 cities and 10 physiographic landscape "
                   "typology (PLT) clusters")
POINT_BASE = "94 marked points (55 hot, 39 cold) from 17 cities and 8 PLT clusters"
ROW_RULE = ("Hit rates are row-based: a point lying within several spot footprints "
            "counts once per footprint, so the denominator grows with the buffer radius.")
NB06_NOTE = ("Source: the NB06 association caches, not the published GeoPackage. The "
             "association analysis is deliberately excluded from the dataset (README "
             "section 7), so this is the one item whose values cannot be traced to it.")

_ITEMS = [
 ("Fig. 1", "Fig_01_study_area", "07 cell 7 (v3.0 Figure 1)",
  f"The 67 Nigerian cities with at least 300,000 inhabitants in 2024 (UN World "
  f"Urbanization Prospects), on the physiographic landscape typology of Nigeria.",
  f"Of these, {len(_profC)} were retained as study cities; 11 lie within the urban core "
  f"of a larger neighbouring city and were merged into it (Section 2.3). The background "
  f"is the national physiographic landscape typology (PLT), all {len(CL)} classes in "
  f"their published colours; each class is named by its three axes - climate, geology "
  f"and elevation. Symbol size gives 2024 city population, the smallest class beginning "
  f"at the 300,000 selection threshold.",
  "`city_profile` (city, cluster25, population_wup_k, absorbed_cities); the WUP table "
  "(positions of the 67 cities and the populations of the 11 absorbed ones; the 56 "
  "study-city populations asserted against `city_profile.population_wup_k`); the "
  "typology background from `landscape_ch_polygons_with_class_metrics.geojson`; the "
  "national boundary."),

 ("Fig. 3", "Fig_03_delineation_excerpt", "07 cell 21 (v3.0 Figure 4)",
  "Three-zone delineation for five cities: Kano, Abuja, Jos, Lagos and Port Harcourt.",
  "Urban core, background reference area (BRA) and city region for each city. The "
  "definition is unchanged from v3.0; only the output name follows the new numbering.",
  "NB01 zone GeoJSONs, as in v3.0."),

 ("Fig. 4", "Fig_04_stakeholder_rings", "07 cell 25 (v3.0 Figure 5)",
  f"Stakeholder participants by physiographic cluster and city: {len(_pcC)} participants "
  f"from {_pcC.city.nunique()} cities in {_pcC.cluster25.nunique()} clusters.",
  "",
  "`stakeholder_perception` (participant_code, city, cluster25), with each participant's "
  "cluster asserted against `city_profile`."),

 ("Fig. 5", "Fig_05_annual_trends", "07 cells 30 + 32 (v3.0 Figure 7)",
  "Annual land surface temperature (LST) by physiographic landscape typology, 2006-2025.",
  "For each of six selected PLT clusters, the annual series. The urban core and "
  "background reference area are read on the left axis and the city-region range is "
  "shaded; the surface urban heat island intensity (SUHII, urban core minus BRA) is read "
  "on its own right-hand axis, because it is a difference of a few degrees against "
  "absolutes of 15 to 45. Day and night are distinguished by colour within each zone, "
  "and dashed lines give the Sen slope. The table below the panels gives the urban-core, "
  "BRA and SUHII trends for both periods from the published cluster composites, with "
  "Mann-Kendall direction and significance.",
  "`composite_trend_statistics` (the table); the NB03 cluster series behind the panels, "
  "asserted equal to the cluster means of `annual_series`, and the NB03 panel statistics "
  "asserted to match the published composites on all 36 slopes."),

 ("Fig. 6", "Fig_06_monthly_trends", "07 cells 30 + 34 (v3.0 Figure 8)",
  "Monthly land surface temperature (LST) climatology by physiographic landscape "
  "typology, 2006-2025.",
  "For the same six clusters, the monthly climatology, drawn as in Fig. 5. The table "
  "below the panels gives the Mann-Kendall direction and significance and the Sen slope "
  "of each cluster's 240-month series for the urban core, BRA and SUHII, by day and night.",
  "the NB03 monthly cluster series and monthly statistics (the package holds annual "
  "composites only); the series asserted equal to the cluster means of `monthly_series`."),

 ("Fig. 7", "Fig_07_heat_event_bars", "07 cell 46 (v3.0 Figure 11)",
  "Persistent heat events of 2024 by physiographic landscape typology.",
  "(a) Mean number of ERA5-Land persistent heat events per city in each cluster, by day "
  "and night. (b) Mean longest event per city. In both, the bar is the cluster mean "
  "across its cities, the error bar spans the lowest and highest city, and the bar border "
  "gives the event tier of that period most common among the cluster's cities. Event "
  "tiers are reported per city in Supplementary Table S4.",
  "`city_year_summary` (ERA5-Land event counts and longest events; "
  "day_selected_event_tier, night_selected_event_tier)."),

 ("Fig. 8", "Fig_08_seviri_suhii_cities", "07 cell 47 (v3.0 Figure 13)",
  "Event-scale surface urban heat island intensity (SUHII) by city.",
  "SUHII (urban core minus BRA) for every city from the MSG-SEVIRI composites over its "
  "selected event window, by day (left) and night (right), sorted by daytime value and "
  "filled by cluster colour; the box gives the national mean and range. Event tiers are "
  "reported per city in Supplementary Table S4.",
  "`city_year_summary` (day_seviri_suhii_c, night_seviri_suhii_c, cluster25)."),

 ("Fig. 9", "Fig_09_spot_maps_day", "07 cell 54 (v3.0 Figure 15)",
  f"Surface hot and cold spots in all {len(_profC)} cities, by day.",
  "Kernel density contours at the 50, 75 and 90 per cent levels (L50, L75, L90) with "
  "tier-2 buffers, over the three zones of each city.",
  "the NB05 A4 sheet and map legend; the spot and contour counts per city asserted "
  "against `spots` and `spot_contours`."),

 ("Fig. 10", "Fig_10_spot_maps_night", "07 cell 54 (v3.0 Figure 16)",
  f"Surface hot and cold spots in all {len(_profC)} cities, at night.",
  "As Fig. 9, for the night-time composites.",
  "as Fig. 9."),

 ("Fig. 11", "Fig_11_spot_area_by_cluster", "07 cell 56 (v3.0 Figure 17)",
  "Spot footprint area by physiographic landscape typology.",
  f"Total spot area per cluster for (a) hot spots by day, (b) hot spots at night, (c) cold "
  f"spots by day and (d) cold spots at night. The three contour levels are NESTED and drawn "
  f"as segments of one bar: L90 at the base, then L75 minus L90, then L50 minus L75, so the "
  f"full bar height is the L50 total. The four panels cover all {len(_spC)} spots.",
  "`spots` (cluster25, spot_type, period, area_l50/l75/l90_km2), the level totals asserted "
  "equal to `spot_contours`."),

 ("Fig. 12", "Fig_12_population_within_spots", "07 cell 62 (v3.0 Figure 20)",
  "Resident population within surface hot and cold spots.",
  f"Moved from the Discussion to Results. National maps with one half-donut per city, "
  f"the day half and the night half scaled by the residents inside the L50 footprints, "
  f"and per-cluster half-donuts beside them. Population is summed from GHS-POP on its "
  f"native 100 m grid, per polygon, with fractional overlap. Nationally, "
  f"{int(round(_hotNightC)):,} residents fall within night-time hot spot footprints at "
  f"L50.",
  "`city_exposure_by_contour`, asserted against the NB05 half-donut summary; "
  "`city_profile` for the city positions."),

 ("Fig. 13", "Fig_13_stakeholder_accounts", "07 cells 38 + 51 (v3.0 Figures 9 + 18)",
  "Stakeholder accounts of urban heat and their agreement with the satellite record.",
  f"(a-c) Perception responses by cluster, as counts of participants with the national "
  f"total as a donut inset: perceived temperature increase over the last ten years, the "
  f"period in which heat is felt most severely, and the months perceived as the "
  f"temperature peak. Based on {PERCEPTION_BASE}. (d) Agreement between "
  f"stakeholder-marked locations and detected spots across buffer radii from 0.5 to "
  f"10 km, hot spots left and cold spots right, with the total hit count shaded on the "
  f"right-hand axis. Based on {POINT_BASE}. {ROW_RULE} Coordinates are the published "
  f"ones, rounded to three decimal degrees, so the curves can be reproduced from the "
  f"dataset alone.",
  "`stakeholder_perception` (the three perception items) and `hitrate_sensitivity` "
  "(buffer_m, n_hit, n_total); the 500 m values are asserted at 4/55, 3/55, 4/39, 0/39."),

 ("Fig. 15", "Fig_15_association_effect_sizes", "07 cell 60 (v3.0 Figure 19)",
  "Association of the physiographic landscape typology with the thermal anomalies, "
  "against its strongest single element.",
  f"Forest plot of effect sizes for each combination of spot type, period and outcome, "
  f"with the full typology against its strongest single element - climate zone, "
  f"geological substrate or elevation belt. Occurrence is tested with a Monte Carlo "
  f"permutation chi-square and reported as Cramer V; SUHII and area are tested with "
  f"Kruskal-Wallis and reported as eta squared. {NB06_NOTE}",
  "None from the package. `Results/06_hotspot_modelling/tables/`."),
]

_TABLES_C = [
 ("Table 1", "Table_01_typology_composition", "07 cell 17 (v3.0 Table 2)",
  f"Physiographic cluster composition of the {len(_profC)} study cities.",
  f"The v3.0 layout: one row per PLT cluster, sorted by the number of cities, with five "
  f"single-city clusters (C6, C8, C15, C21, C23) grouped into a final row. Columns give "
  f"the member cities, the city count, the physiographic description (climate · geology "
  f"· elevation), the summed 2024 WUP population after absorption, and the summed "
  f"city-region and urban-core area, computed in ESRI:102022. C13 alone holds "
  f"{_share13C:.1f} per cent of the urban-core area of all {len(_profC)} study cities.",
  "`city_profile` (cluster25, city, population_wup_merged_k, cr_area_km2, uc_area_km2)."),

 ("Table 2", "Table_02_significant_spots_by_cluster", "07 cell 52 (v3.0 Table 5)",
  "Significant spots by physiographic landscape typology.",
  "Spot counts, contributing cities and total L50 footprint area for each cluster, by "
  "spot type and period, with a national total row. Benjamini-Hochberg FDR survival is "
  "a property of the city-period-type group, repeated on every spot of that group, not "
  "a per-spot test.",
  "`spots` (cluster25, city, period, spot_type, area_l50_km2, bh_significant), through "
  "the v3.0 builder in `src/pub_hotspot_tables.py`."),

 ("Table 3", "Table_03_stakeholder_vs_satellite_cluster", "07 cell 39 (v3.0 Table 4)",
  "Stakeholder accounts against the satellite record, by physiographic landscape "
  "typology.",
  f"One row per cluster, based on {PERCEPTION_BASE}. Every column rests on the same "
  f"per-city basis: the trend columns count the member cities with stakeholders whose "
  f"urban-core Mann-Kendall trend is a significant increase (p < 0.05), and the "
  f"agreement columns are computed city by city. Nationally that is 12 of 21 cities by "
  f"day and 14 of 21 by night. Seven cities are represented by a single stakeholder "
  f"({', '.join(_singlesC)}) and are flagged, because one account cannot express "
  f"agreement within a city. Nationally, 70 of 105 participants named a window "
  f"containing the observed hottest month. City rows, each with its own trend, are in "
  f"Supplementary Table S6.",
  "`stakeholder_perception` and `trend_statistics` (zone = uc); the agreement columns "
  "from `Stakeholder_Validation_Comprehensive.csv`."),

 ("Table 4", "Table_04_typology_association", "07 cells 52 + 58 (v3.0 Tables 6 + 7)",
  "Association of the typology and its elements with the thermal anomalies.",
  f"Spot type by period by outcome, giving the test applied, the typology effect size "
  f"and p-value, the strongest single element with its effect size and Holm-adjusted "
  f"p-value, and the difference between them. Five of the twelve typology tests reach "
  f"p < 0.05. {NB06_NOTE}",
  "None from the package. `Results/06_hotspot_modelling/tables/`."),
]

_nl = chr(10)
_out = ["# Captions - SCS main figures and tables", "",
        "Generated by cell %d of `Notebook/07_publication_outputs.ipynb`, so the counts "
        "stay tied to the package." % 64, "",
        "Figures carry no titles of their own; these captions carry them. Panel letters "
        "stay on the figures. Cluster names follow "
        "`Results/manuscript_outputs/cluster_labels.csv`, and typology colours are the "
        "published ones from `config.CLUSTER25_COLORS`.", "",
        "**Two stakeholder denominators, never mixed.** Perception items rest on "
        + PERCEPTION_BASE + ". Point-based items rest on " + POINT_BASE + ", because "
        "four cohort cities answered the perception items but marked no locations.", "",
        "**Event tier.** `validation_tier` and the selected-event tiers are a day/night "
        "pair, written as event tier (day / night), for example EXCELLENT/GOOD. A single "
        "event carries one grade, written as event tier with the period of that event.",
        "", "---", ""]
for _num, _stem, _origin, _title, _body, _src in _ITEMS:
    _out += [f"## {_num} - {_title}", "",
             f"`main_figures/{_stem}.(pdf|svg)`  ·  from {_origin}", "",
             f"**{_num}.** {_title} {_body}", "", f"*Reads:* {_src}", "", "---", ""]
for _num, _stem, _origin, _title, _body, _src in _TABLES_C:
    _out += [f"## {_num} - {_title}", "",
             f"`tables/{_stem}.(csv|xlsx)`  ·  from {_origin}", "",
             f"**{_num}.** {_title} {_body}", "", f"*Reads:* {_src}", "", "---", ""]
_out += ["## Supplementary items referenced above", "",
         f"- Supplementary Figure S12: city-level perception for the {len(_multiC)} "
         f"cities with at least two participants; that threshold omits "
         f"{len(_droppedC)} cities ({', '.join(_droppedC)}).",
         "- Supplementary Table S4: selected events and their event tiers, per city.",
         f"- Supplementary Table S6: stakeholder accounts per city, each with its own "
         f"trend, with the {len(_singlesC)} single-stakeholder cities flagged.",
         f"- Supplementary Table S7: hit rate by city and buffer, on the same "
         f"{POINT_BASE}. {ROW_RULE}", ""]
(OUT / "captions.md").write_text(_nl.join(_out), encoding="utf-8")
print("captions.md:", len(_ITEMS), "figures,", len(_TABLES_C), "tables")

# ── contact sheet, through the checked savefig ──
_pdfs = sorted(p for p in MAIN_FIG.glob("Fig_*.pdf"))
if _pdfs:
    _cols = 3; _rows = int(np.ceil(len(_pdfs) / _cols))
    figC, _axC = plt.subplots(_rows, _cols, figsize=(8.27, 11.69))
    _axC = np.atleast_1d(_axC).ravel()
    for _a in _axC:
        _a.set_axis_off()
    for _a, _p in zip(_axC, _pdfs):
        _doc = _fitzC.open(_p); _pm = _doc[0].get_pixmap(dpi=96)
        _img = np.frombuffer(_pm.samples, dtype=np.uint8).reshape(_pm.height, _pm.width, _pm.n)
        _a.imshow(_img[:, :, :3]); _a.set_title(_p.stem.replace("_", " "), fontsize=7)
        _doc.close()
    figC.suptitle("SCS main figures — contact sheet (%d items, %s)"
                  % (len(_pdfs), pd.Timestamp.today().date().isoformat()),
                  fontsize=10, fontweight="bold")
    figC.tight_layout(rect=(0, 0, 1, 0.97))
    _pC = savefig(figC, MAIN_FIG / "_thumbnails_main_figures")
    plt.show()
    print("Contact sheet:", len(_pdfs), "figures ->", [p.name for p in _pC])

# ── print size and smallest type, measured on the written PDFs ──────────
_SZ = fig_size_report()
print("" + chr(10) + "Print size and smallest type (target: width <= 190 mm, "
      "height <= 240 mm, text >= %.1f pt)" % MIN_PT)
print(_SZ.to_string(index=False))
_bad_sz = _SZ[~_SZ.ok]
print("Reported only (plan 1a.5), never a reason to change a v3.0 layout:",
      f"{len(_SZ)} figures; widest {_SZ.width_mm.max():.0f} mm, tallest "
      f"{_SZ.height_mm.max():.0f} mm, smallest type {_SZ.min_pt.min():.1f} pt"
      + ("" if len(_bad_sz) == 0 else "; failing: "
         + ", ".join(f"{r.figure} ({r.width_mm:.0f}x{r.height_mm:.0f} mm, "
                     f"{r.min_pt:.1f} pt)" for r in _bad_sz.itertuples())))
_SZ.to_csv(OUT / "figure_print_sizes.csv", index=False)

# ── every published file must really be the format its extension claims ──
_badC = [p.name for p in sorted(MAIN_FIG.glob("*.pdf"))
         if p.open("rb").read(5) != b"%PDF-"]
_badC += [p.name for p in sorted(MAIN_FIG.glob("*.svg"))
          if p.open("rb").read(5) != b"<?xml"]
check("every figure file matches its extension", not _badC,
      f"{len(list(MAIN_FIG.glob('*.*')))} files checked" + (f"; bad: {_badC}" if _badC else ""))

In [ ]:
begin("Assembly")
# ── supplementary contact sheet: the first page of every set ─────────────────
_sets = []
for _d in sorted(p for p in SUP_FIG.iterdir() if p.is_dir()):
    _f = sorted(list(_d.rglob("*.pdf")) + list(_d.rglob("*.svg")))
    if _f:
        _sets.append((_d.name, _f[0]))
_cols = 4
figS, _axS = plt.subplots(int(np.ceil(len(_sets) / _cols)), _cols, figsize=(8.27, 11.69))
for _a in np.atleast_1d(_axS).ravel():
    _a.set_axis_off()
for _a, (_nm, _p) in zip(np.atleast_1d(_axS).ravel(), _sets):
    _doc = _as_page(_p)
    _pm = _doc[0].get_pixmap(dpi=60, alpha=False)
    _a.imshow(np.frombuffer(_pm.samples, dtype=np.uint8).reshape(_pm.height, _pm.width, _pm.n))
    _a.set_title(f"{_nm}\n{_p.name}", fontsize=6.5)
    _doc.close()
figS.tight_layout()
savefig(figS, SUP_FIG / "_thumbnails_supplementary", fit=None)
plt.close(figS)
print(f"Supplementary contact sheet: {len(_sets)} sets")

# ── old-to-new number map (v3.0 -> SCS v4, plan v3.2) ────────────────────────
NUMBER_MAP = pd.DataFrame([
    ("Figure 1", "Fig. 1"), ("Figure 2", "Supp. Fig. S1.0"), ("Figure 3", "Fig. 2 (author)"),
    ("Figure 4", "Fig. 3"), ("Figure 5", "Fig. 4"), ("Figure 6", "removed (DiB Fig. 3)"),
    ("Figure 7", "Fig. 5"), ("Figure 8", "Fig. 6"), ("Figure 9", "Fig. 13"),
    ("Figure 10", "Supp. Fig. S5.0"), ("Figure 11", "Fig. 7"), ("Figure 12", "removed (Supp. Fig. S7)"),
    ("Figure 13", "Fig. 8"), ("Figure 14", "Supp. Fig. S8.0"), ("Figure 15", "Fig. 9"),
    ("Figure 16", "Fig. 10"), ("Figure 17", "Fig. 11"), ("Figure 18", "Fig. 14"),
    ("Figure 19", "Fig. 15"), ("Figure 20", "Fig. 12"),
    ("Table 1", "removed (DiB)"), ("Table 2", "Table 1"), ("Table 3", "removed (DiB)"),
    ("Table 4", "Table 3"), ("Table 5", "Table 2"), ("Table 6", "Table 4"), ("Table 7", "Table 5"),
    ("Supp. Fig. 1.1.x", "Supp. Fig. S1.1.x"), ("Supp. Fig. 1.2.x", "Supp. Fig. S1.2.x"),
    ("Supp. Fig. 2.x", "Supp. Fig. S2.x"), ("Supp. Fig. 3.1", "Supp. Fig. S3.1"),
    ("Supp. Fig. 3.2", "Supp. Fig. S3.2"), ("Supp. Fig. 4.x and 5.x", "Supp. Fig. S4"),
    ("Supp. Fig. 6", "Supp. Fig. S11"), ("Supp. Fig. 7.1 / 7.2 / 7.3", "Supp. Fig. S5.1 / S5.2 / S5.3"),
    ("Supp. Fig. 8.1 / 8.2", "Supp. Fig. S6"), ("Supp. Fig. 9", "Supp. Fig. S7.1"),
    ("(DiB Fig. 5)", "Supp. Fig. S7.2"), ("Supp. Fig. 10.x", "Supp. Fig. S8.x"),
    ("Supp. Fig. 11.1 / 11.2", "Supp. Fig. S9.1 / S9.2"), ("Supp. Fig. 12.x", "Supp. Fig. S10"),
    ("Appendix I", "Supp. Table S1"), ("Supp. Table 1", "removed"),
    ("Supp. Table 2", "Supp. Table S2"), ("Supp. Table 3", "Supp. Table S6"),
    ("Supp. Table 4", "Supp. Table S3"), ("Supp. Table 5", "Supp. Table S4"),
    ("Supp. Table 6", "Supp. Table S5"), ("Supp. Table 7", "Supp. Table S7"),
    ("Supp. Table 8", "Supp. Table S8"),
], columns=["v3.0", "SCS v4"])
NUMBER_MAP.to_csv(OUT / "number_map.csv", index=False)
print(f"number_map.csv: {len(NUMBER_MAP)} rows")

# ── provenance summary, from _data_sources.csv ───────────────────────────────
_src = pd.DataFrame(SOURCE_ROWS)
_oos = _src[_src.permitted == "OOS"].groupby("item")["category"].apply(
    lambda s: ", ".join(sorted(set(s))))
print(f"\n{len(_src)} reads logged in {SOURCES_CSV.name}: "
      + ", ".join(f"{k} {v}" for k, v in _src.category.value_counts().items()))
print("Reads outside the item's permitted sources (to move to the package in that item's step):")
print(_oos.to_string() if len(_oos) else "  none")
check("every read passed the source guard (no forbidden data source)", True,
      f"{len(_src)} reads, {_src.path.nunique()} distinct files or folders")

# ── forbidden-path scan of this notebook's own code ──────────────────────────
_nbfile = ROOT / "Notebook" / "07_publication_outputs.ipynb"
_hits = [f"cell {i}: {ln.strip()[:80]}"
         for i, c in enumerate(json.loads(_nbfile.read_text(encoding="utf-8"))["cells"])
         if c["cell_type"] == "code"
         for ln in "".join(c["source"]).splitlines()
         if "forbidden-list" not in ln and any(f in ln for f in FORBIDDEN_PARTS)]
check("the notebook's code names no forbidden data path", not _hits, "; ".join(_hits[:5]))

# ── side-by-side previews and the generated part of INDEX.md ─────────────────
# (No., title, section, v3.0 reference, current output). Outputs keep their C1 names until
# each item's step gives them the new number.
ITEMS = [
    ("Fig. 1", "Study area", "2.1", "main_figures/Figure_1_study_area.pdf", "main_figures/Fig_01_study_area.pdf"),
    ("Fig. 3", "Delineation excerpt", "2.3", "main_figures/Figure_4_city_aoi_grid.pdf", "main_figures/Fig_03_delineation_excerpt.pdf"),
    ("Table 1", "Typology composition", "2.3", ROOT / "docs/scs_rebuild/reference/Table1_v3_manuscript.png", "tables/Table_01_typology_composition.csv"),
    ("Fig. 4", "Stakeholder rings", "2.3", "main_figures/Figure_5_stakeholder_radial.pdf", "main_figures/Fig_04_stakeholder_rings.pdf"),
    ("Fig. 5", "Annual trends", "3.1", "main_figures/Figure_7_annual_trend.pdf", "main_figures/Fig_05_annual_trends.pdf"),
    ("Fig. 6", "Monthly trends", "3.1", "main_figures/Figure_8_monthly_trend.pdf", "main_figures/Fig_06_monthly_trends.pdf"),
    ("Fig. 7", "Persistent heat events", "3.1", "main_figures/Figure_11_heat_event_bars.pdf", "main_figures/Fig_07_heat_event_bars.pdf"),
    ("Fig. 8", "Event-scale SUHII by city", "3.1", "main_figures/Figure_13_seviri_suhii_cities.pdf", "main_figures/Fig_08_seviri_suhii_cities.pdf"),
    ("Table 2", "Significant spots by cluster", "3.2", "tables/Table_5_cluster_spot_occurrence.csv", "tables/Table_02_significant_spots_by_cluster.csv"),
    ("Fig. 9", "Spots, all cities, day", "3.2", "main_figures/Figure_15_spatial_extent_day.pdf", "main_figures/Fig_09_spot_maps_day.pdf"),
    ("Fig. 10", "Spots, all cities, night", "3.2", "main_figures/Figure_16_spatial_extent_night.pdf", "main_figures/Fig_10_spot_maps_night.pdf"),
    ("Fig. 11", "Spot area by cluster", "3.2", "main_figures/Figure_17_zone_area_distribution.pdf", "main_figures/Fig_11_spot_area_by_cluster.pdf"),
    ("Fig. 12", "Population within spots", "3.3", "main_figures/Figure_20_population_exposure_panel.pdf", "main_figures/Fig_12_population_within_spots.pdf"),
    ("Table 3", "Stakeholder accounts vs satellite", "3.4", "tables/Table_4_stakeholder_corroboration.csv", "tables/Table_03_stakeholder_vs_satellite_cluster.csv"),
    ("Fig. 13", "Stakeholder perception", "3.4", "main_figures/Figure_9_stakeholder_perception.svg", "main_figures/Fig_13_stakeholder_accounts.pdf"),
    ("Fig. 14", "Hit rate vs buffer", "3.4", "main_figures/Figure_18_stakeholder_hitrate_national.svg", "main_figures/Fig_13_stakeholder_accounts.pdf"),
    ("Table 4", "Typology-anomaly association", "3.5", "tables/Table_6_plt_thermal_association.csv", "tables/Table_04_typology_association.csv"),
    ("Table 5", "Best single-element attribution", "3.5", "tables/Table_7_best_element_attribution_excerpt.csv", "tables/Table_7_best_element_attribution_excerpt.csv"),
    ("Fig. 15", "Effect sizes (forest plot)", "3.5", "main_figures/Figure_19_forest_plot.pdf", "main_figures/Fig_15_association_effect_sizes.pdf"),
] + [(f"S{k}", t, s, None, o) for k, t, s, o in [
    (1, "Population trajectories", "2.1", "supplementary_figures/1.1_city_trends"),
    (2, "City delineation maps", "2.3", "supplementary_figures/2_city_aoi_maps"),
    (3, "Stakeholder sample maps", "2.3", "supplementary_figures/S3_stakeholder_maps"),
    (4, "Long-term trends, complete", "3.1", "supplementary_figures/4_temporal_trends"),
    (5, "Daily series and events", "3.1", "supplementary_figures/7_daily_phe"),
    (6, "Event frequency by cluster", "3.1", "supplementary_figures/8_event_frequency"),
    (7, "Thermal-product agreement", "3.1", "supplementary_figures/9_hexbin"),
    (8, "Gi* z-score distributions", "3.2", "supplementary_figures/10_gistar_distributions"),
    (9, "City spot area, ranked", "3.2", "supplementary_figures/11_city_area_ranked"),
    (10, "Population within spots, full set", "3.3", "supplementary_figures/12_population_exposure"),
    (11, "City-level perception", "3.4", "supplementary_figures/S12_city_perception")]] + [
    (f"Supp. Table S{k}", t, s, None, o) for k, t, s, o in [
    (1, "24 typologies", "2.1", "appendices/Appendix_1_PLT_reference.csv"),
    (2, "Normality tests", "2.3", None),
    (3, "Heat events per city", "3.1", "tables/Supp_Table_4_event_frequency.csv"),
    (4, "Selected events, sensor agreement", "3.1", "tables/Supp_Table_5_selected_events_correlations.csv"),
    (5, "Spots and contours per city", "3.2", "tables/Supp_Table_S5_spots_per_city.csv"),
    (6, "Stakeholder accounts per city", "3.4", "tables/Supp_Table_S6_stakeholder_per_city.csv"),
    (7, "Hit rate by city and buffer", "3.4", "tables/Supp_Table_S7_hit_rate_by_city.csv"),
    (8, "Best-element attribution, complete", "3.5", "tables/Supp_Table_S8_best_element_attribution.csv")]]

_cells = {}
for i, c in enumerate(json.loads(_nbfile.read_text(encoding="utf-8"))["cells"]):
    for m in re.finditer(r'^begin\("([^"]+)"\)', "".join(c["source"]), re.M):
        _cells.setdefault(m.group(1), []).append(str(i))
_rows = []
for no, title, sect, ref, out in ITEMS:
    _out = OUT / out if out else None
    _prev = ""
    if ref:
        _prev = f"[{no}]({compare_preview(no, V3 / ref, _out).name})"
    _size = ""
    if _out is not None and _out.suffix == ".pdf" and _out.exists():
        w, h, m = fig_report(_out, no)
        rw, rh, rm = fig_report(V3 / ref, f"{no} (v3.0)")
        _size = f"{w}×{h}, {m if m else 'n/m'} ({rw}×{rh}, {rm if rm else 'n/m'})"
    _s = _src[_src.item == no]
    _ds = "; ".join(f"{cat}{' **OOS**' if (g.permitted == 'OOS').any() else ''} ×{len(g)}"
                    + (f" ({', '.join(sorted(set(g.layer) - {''}))})" if cat == "PKG" else "")
                    for cat, g in _s.groupby("category")) or "—"
    _a = [a for a in ASSERTIONS if a[3] == no]
    _rows.append(f"| {no} | {title} | {sect} | {', '.join(_cells.get(no, [])) or '—'} | "
                 f"{'`' + out + '`' if out else '—'}{'' if (_out is None or _out.exists()) else ' (missing)'} | "
                 f"{_prev or '—'} | {_ds} | {sum(1 for a in _a if a[1])}/{len(_a)} | {_size or '—'} |")
_auto = ["<!-- AUTO:BEGIN — written by the Assembly section of Notebook 07; do not edit by hand -->",
         "", f"Generated {pd.Timestamp.now():%Y-%m-%d %H:%M}. Cells are the notebook's code cells "
         "that call `begin()` for the item; data sources come from `_data_sources.csv` "
         "(category ×reads; **OOS** = not permitted for this item by plan Section 3); checks are "
         "passed/total `check()` assertions tagged with the item; size is width × height mm, "
         "smallest font pt, new (v3.0).", "",
         "| No. | Title | Sect. | Cells | Current output | Preview | Data sources | Checks | Size, min font: new (v3.0) |",
         "|---|---|---|---|---|---|---|---|---|"] + _rows + ["", "<!-- AUTO:END -->"]
_idx = PREVIEW / "INDEX.md"
_txt = _idx.read_text(encoding="utf-8") if _idx.exists() else ""
_i, _j = _txt.find("<!-- AUTO:BEGIN"), _txt.find("<!-- AUTO:END -->")
_txt = (_txt[:_i] + "\n".join(_auto) + _txt[_j + len("<!-- AUTO:END -->"):] if _i >= 0 and _j > _i
        else "# SCS figures and tables — monitoring index\n\n" + "\n".join(_auto) + "\n" + _txt)
_idx.write_text(_txt, encoding="utf-8", newline="\n")
print(f"INDEX.md: {len(_rows)} items; previews for "
      f"{sum(1 for r in ITEMS if r[3])} main items in {PREVIEW.relative_to(ROOT)}")

# ── assertion summary ────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("Assertions: %d of %d passed" % (sum(1 for a in ASSERTIONS if a[1]), len(ASSERTIONS)))
for _lbl, _ok, _d, _it in ASSERTIONS:
    print(("  [OK]   " if _ok else "  [FAIL] ") + f"[{_it}] " + _lbl + (f" - {_d}" if _d else ""))
print("=" * 72)